This isn't part of the automated pipeline, but needed to get input data ready. It may vary depending on the data source.


# Extract and merge Gitter data

In [ ]:
import os
import zipfile
from doctest import debug

import pandas as pd
import re


# To use for all three data levels, find and replace '100m' with '1km' or '10km' as needed.

def extract_zip_files(zip_folder, extract_to):
    """Extracts all zip files in the given folder."""
    zip_files = [f for f in os.listdir(zip_folder) if f.endswith(".zip")]
    print(f"Found {len(zip_files)} zip files.")

    extracted_files = []
    for zip_file in zip_files:
        zip_path = os.path.join(zip_folder, zip_file)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
            extracted_files.extend([os.path.join(extract_to, name) for name in z.namelist()])

    print(f"Extracted {len(extracted_files)} files.")
    return extracted_files


def find_csv_files(files):
    """Finds CSV files that contain '1km-Gitter' in the name."""
    csv_files = [f for f in files if "1km-Gitter" in f and f.endswith(".csv")]
    print(f"Found {len(csv_files)} relevant CSV files.")
    return csv_files


def clean_text(text):
    """Removes special characters and replaces them with an empty string."""
    return re.sub(r'[^\x20-\x7E]', '', text)  # Keeps only printable ASCII characters


def convert_numeric_columns(df):
    """Ensures numeric columns with comma decimals ('9,07') are properly converted to floats."""
    for col in df.columns:
        # Only modify string columns
        if df[col].dtype == object:
            df[col] = df[col].str.replace(',', '.', regex=False)  # Replace commas with dots for decimal conversion
            df[col] = pd.to_numeric(df[col], errors='ignore')  # Convert to float, set invalid entries to NaN

    return df


def read_csv_with_encoding(file):
    """Reads CSV with UTF-8 first, falls back to ISO-8859-1 if needed."""
    try:
        df = pd.read_csv(file, encoding='utf-8', dtype=str, sep=';', encoding_errors='replace')
    except UnicodeDecodeError:
        print(f"Warning: UTF-8 failed for {file}, trying ISO-8859-1.")
        df = pd.read_csv(file, encoding='ISO-8859-1', dtype=str, sep=';', encoding_errors='replace')

    # Apply text cleaning to all string values
    df = df.applymap(lambda x: clean_text(x) if isinstance(x, str) else x)

    # Convert numeric values from '9,07' to 9.07
    df = convert_numeric_columns(df)

    return df


def process_and_merge_csv(files):
    """Reads, processes, and merges CSV files on 'GITTER_ID_1km'."""
    merged_df = None

    for file in files:
        print(f"Processing: {file}")
        df = read_csv_with_encoding(file)

        # Standardize column names
        df.columns = df.columns.str.strip()

        # Drop unwanted columns if they exist
        df.drop(columns=[col for col in ['x_mp_1km', 'y_mp_1km'] if col in df.columns], inplace=True, errors='ignore')

        if 'GITTER_ID_1km' not in df.columns:
            print(f"Warning: 'GITTER_ID_1km' not found in {file}. Available columns: {df.columns.tolist()}")
            continue

        # Rename duplicate columns before merging
        df = df.rename(columns={col: f"{col}_{os.path.basename(file).split('.')[0]}" for col in df.columns if
                                col != 'GITTER_ID_1km'})

        if merged_df is None:
            merged_df = df
        else:
            merged_df = merged_df.merge(df, on='GITTER_ID_1km', how='outer', suffixes=(None, None))

    print("Merging complete.")
    return merged_df


# Define paths
zip_folder = r"C:\Users\petre\Desktop\Synpop\Zensus"
extract_to = r"C:\Users\petre\Desktop\Synpop\Zensus\Extrahiert"
output_file = "merged_1km_gitter.csv"
output_excel = "merged_1km_gitter.xlsx"

# Execution pipeline
extracted_files = extract_zip_files(zip_folder, extract_to)
csv_files = find_csv_files(extracted_files)
merged_data = process_and_merge_csv(csv_files)

# Save to CSV
if merged_data is not None:
    merged_data.to_csv(output_file, index=False)
    print(f"Saved merged file to {output_file}")
else:
    print("No valid data to save.")



# Impute missing/inaccurate population values (just multiplication to fit the total -> very low values stay very low)

In [38]:
import pandas as pd
import re


def normalize_id(gid, replace_digits):
    match = re.search(r'(N\d+)(E\d+)', gid)
    if match:
        n_part, e_part = match.groups()
        n_base = n_part[:-replace_digits] + '0' * replace_digits
        e_base = e_part[:-replace_digits] + '0' * replace_digits
        return n_base + e_base
    return gid


def smart_integerize(df, pop_cols):
    for pop_col in pop_cols:
        total_before = df[pop_col].sum()
        df[pop_col] = df[pop_col].round().astype(int)
        total_after = df[pop_col].sum()
        diff = total_before - total_after
        if diff != 0:
            sorted_indices = df[pop_col].abs().sort_values(ascending=False).index[:abs(int(diff))]
            df.loc[sorted_indices, pop_col] += int(diff / abs(diff))


def adjust_population(lower_level_file, higher_level_file, replace_digits_lower, replace_digits_higher, id_col_lower,
                      id_col_higher, pop_cols_lower, pop_cols_higher, output_file):
    lower_df = pd.read_csv(lower_level_file)
    higher_df = pd.read_csv(higher_level_file)

    assert id_col_lower in lower_df.columns and all(
        col in lower_df.columns for col in pop_cols_lower), "Lower level dataset missing required columns."
    assert id_col_higher in higher_df.columns and all(
        col in higher_df.columns for col in pop_cols_higher), "Higher level dataset missing required columns."

    lower_df['GroupID'] = lower_df[id_col_lower].apply(lambda x: normalize_id(str(x), replace_digits_lower))
    higher_df['GroupID'] = higher_df[id_col_higher].apply(lambda x: normalize_id(str(x), replace_digits_higher))

    lower_grouped = lower_df.groupby('GroupID')[pop_cols_lower].sum().reset_index()
    merged = lower_grouped.merge(higher_df[['GroupID'] + pop_cols_higher], on='GroupID', how='left')

    for pop_lower, pop_higher in zip(pop_cols_lower, pop_cols_higher):
        merged[pop_higher] = merged[pop_higher].fillna(merged[pop_lower])
        merged[f'AdjustmentFactor_{pop_higher}'] = merged[pop_higher] / merged[pop_lower]
        merged[f'AdjustmentFactor_{pop_higher}'].fillna(1.0, inplace=True)

    lower_df = lower_df.merge(merged[['GroupID'] + [f'AdjustmentFactor_{col}' for col in pop_cols_higher]],
                              on='GroupID', how='left')

    for pop_lower, pop_higher in zip(pop_cols_lower, pop_cols_higher):
        lower_df[f'Adjusted_{pop_lower}'] = lower_df[pop_lower] * lower_df[f'AdjustmentFactor_{pop_higher}']

    # Set nans to 0
    lower_df.fillna(0, inplace=True)
    # This ensures total population, but within the cell, it may cause minor differences (since this is global)
    smart_integerize(lower_df, [f'Adjusted_{col}' for col in pop_cols_lower])

    save_csv = lower_df[[id_col_lower] + [f'Adjusted_{col}' for col in pop_cols_lower]]
    # Remove the adjusted prefix in the col name
    save_csv.columns = [col.replace('Adjusted_', '') for col in save_csv.columns]
    save_csv.to_csv(output_file, index=False)

    print(f"Adjusted population data saved to {output_file}")


adjust_population(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.csv",
                  r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.csv",
                  replace_digits_lower=4, replace_digits_higher=4,
                  id_col_lower='GITTER_ID_1km', id_col_higher='GITTER_ID_10km',
                  pop_cols_lower=['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter',
                                  'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'],
                  pop_cols_higher=['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
                                   'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'],
                  output_file="1adjusted_1km.csv")

adjust_population(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.csv",
                  "1adjusted_1km.csv", replace_digits_lower=3, replace_digits_higher=3,
                  id_col_lower='GITTER_ID_100m', id_col_higher='GITTER_ID_1km',
                  pop_cols_lower=['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter',
                                  'Insgesamt_Bevoelkerung_Zensus2022_Alter_5Altersklassen_100m-Gitter'],
                  pop_cols_higher=['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter',
                                   'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'],
                  output_file="1adjusted_100m.csv")


C:\Users\petre\AppData\Local\Temp\ipykernel_15232\3029198941.py:24: DtypeWarning: Columns (27,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  lower_df = pd.read_csv(lower_level_file)
C:\Users\petre\AppData\Local\Temp\ipykernel_15232\3029198941.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged[f'AdjustmentFactor_{pop_higher}'].fillna(1.0, inplace=True)
C:\Users\petre\AppData\Local\Temp\ipykernel_15232\3029198941.py:39: FutureWarning: A value is trying to be set on a copy of a DataFra

Adjusted population data saved to adjusted_100m.csv


# Merge and do some cleanup

In [42]:
import pandas as pd


def merge_adjusted_with_original(original_file, adjusted_file, pop_cols, to_file):
    original_df = pd.read_csv(original_file)
    adjusted_df = pd.read_csv(adjusted_file)

    for pop_col in pop_cols:
        total_diff = adjusted_df[pop_col].sum() - original_df[pop_col].sum()
        max_diff = (adjusted_df[pop_col] - original_df[pop_col]).abs().max()
        print(f"{pop_col} - Total Difference: {total_diff}, Max Difference per row: {max_diff}")

    for pop_col in pop_cols:
        original_df[f'adjustment_diff_{pop_col}'] = adjusted_df[pop_col] - original_df[pop_col]
        original_df[pop_col] = adjusted_df[pop_col]

    if "Insgesamt_Bevoelkerung_Zensus2022_Alter_5Altersklassen_100m-Gitter" in original_df.columns:
        original_df.rename(columns={
            "Insgesamt_Bevoelkerung_Zensus2022_Alter_5Altersklassen_100m-Gitter": "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_100m-Gitter"},
            inplace=True)

    original_df.to_csv(to_file, index=False)
    print(f"Merged data saved to {to_file}")


merge_adjusted_with_original(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.csv",
                             "adjusted_1km.csv", [
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter',
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'
                             ],
                             "2poptotal_adjusted_1km_gitter.csv")

merge_adjusted_with_original(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.csv",
                             "adjusted_100m.csv", [
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter',
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_5Altersklassen_100m-Gitter'
                             ],
                             "2poptotal_adjusted_100m_gitter.csv")

C:\Users\petre\AppData\Local\Temp\ipykernel_15232\967282453.py:4: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  original_df = pd.read_csv(original_file)


Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter - Total Difference: 4921.0, Max Difference per row: 13.0
Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter - Total Difference: 4921.0, Max Difference per row: 13.0
Merged data saved to poptotal_adjusted_1km_gitter.csv


C:\Users\petre\AppData\Local\Temp\ipykernel_15232\967282453.py:4: DtypeWarning: Columns (27,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  original_df = pd.read_csv(original_file)


Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter - Total Difference: 135944.0, Max Difference per row: 12.0
Insgesamt_Bevoelkerung_Zensus2022_Alter_5Altersklassen_100m-Gitter - Total Difference: 135944.0, Max Difference per row: 12.0
Merged data saved to poptotal_adjusted_100m_gitter.csv


# Ensure the population totals match

In [45]:
compare1 = pd.read_csv("2poptotal_adjusted_1km_gitter.csv")
compare2 = pd.read_csv("2poptotal_adjusted_100m_gitter.csv")
diff = compare1["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter"] - compare1[
    "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter"]
print(max(diff))
diff = compare2["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter"] - compare2[
    "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_100m-Gitter"]
print(max(diff))

C:\Users\petre\AppData\Local\Temp\ipykernel_15232\2574285654.py:1: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  compare1 = pd.read_csv("poptotal_adjusted_1km_gitter.csv")
C:\Users\petre\AppData\Local\Temp\ipykernel_15232\2574285654.py:2: DtypeWarning: Columns (27,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  compare2 = pd.read_csv("poptotal_adjusted_100m_gitter.csv")


0
0


# Impute age distribution

In [31]:
# Add higher level IDs to 1km and 100m

def normalize_id(gid, replace_digits):
    match = re.search(r'(N\d+)(E\d+)', gid)
    if match:
        n_part, e_part = match.groups()
        n_base = n_part[:-replace_digits] + '0' * replace_digits
        e_base = e_part[:-replace_digits] + '0' * replace_digits
        return n_base + e_base
    return gid


def add_higher_level_id(df, id_col, replace_digits, new_col_name):
    df[new_col_name] = df[id_col].apply(lambda x: normalize_id(str(x), replace_digits))
    return df


def prepend_id(df, id_col, prefix, new_col_name):
    df[new_col_name] = prefix + df[id_col].astype(str)
    return df


# Add 10km ID to 1km
one_km_df = pd.read_csv("2poptotal_adjusted_1km_gitter.csv")
one_km_df = add_higher_level_id(one_km_df, 'GITTER_ID_1km', 4, 'GITTER_ID_10km')
one_km_df = prepend_id(one_km_df, 'GITTER_ID_10km', 'CRS3035RES10000m', 'GITTER_ID_10km')
one_km_df.to_csv("3adjusted_1km_with_higher.csv", index=False)

# Add 1km ID to 100m
hundred_m_df = pd.read_csv("2poptotal_adjusted_100m_gitter.csv")
hundred_m_df = add_higher_level_id(hundred_m_df, 'GITTER_ID_100m', 3, 'GITTER_ID_1km')
hundred_m_df = prepend_id(hundred_m_df, 'GITTER_ID_1km', 'CRS3035RES1000m', 'GITTER_ID_1km')

# Add 10km ID to 100m
hundred_m_df = add_higher_level_id(hundred_m_df, 'GITTER_ID_100m', 4, 'GITTER_ID_10km')
hundred_m_df = prepend_id(hundred_m_df, 'GITTER_ID_10km', 'CRS3035RES10000m', 'GITTER_ID_10km')
hundred_m_df.to_csv("3adjusted_100m_with_higher.csv", index=False)


C:\Users\petre\AppData\Local\Temp\ipykernel_17900\3198254823.py:21: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  one_km_df = pd.read_csv("2poptotal_adjusted_1km_gitter.csv")
C:\Users\petre\AppData\Local\Temp\ipykernel_17900\3198254823.py:27: DtypeWarning: Columns (27,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  hundred_m_df = pd.read_csv("2poptotal_adjusted_100m_gitter.csv")


In [4]:
import pandas as pd
import numpy as np
import re


def smart_integerize(df, pop_cols):
    for pop_col in pop_cols:
        total_before = df[pop_col].sum()
        df[pop_col] = df[pop_col].round().astype(int)
        total_after = df[pop_col].sum()
        diff = total_before - total_after
        if diff != 0:
            sorted_indices = df[pop_col].abs().sort_values(ascending=False).index[:abs(int(diff))]
            df.loc[sorted_indices, pop_col] += int(diff / abs(diff))
    return df


def ipf_adjustment(df, row_control_totals, col_control_totals, age_cols, max_iterations=10, tol=1e-6):
    for _ in range(max_iterations):
        
        # We do col total matching last here, because we are interested, at 10km in the best age DISTRIBUTION, not exact matching total.
        row_totals = df[age_cols].sum(axis=1)
        scaling_factors = row_control_totals / row_totals
        # Set inf to 1
        scaling_factors[~np.isfinite(scaling_factors)] = 1
        df[age_cols] = df[age_cols].multiply(scaling_factors, axis=0)

        col_totals = df[age_cols].sum()
        scaling_factors = col_control_totals / col_totals
        # Set inf to 1
        scaling_factors[~np.isfinite(scaling_factors)] = 1
        df[age_cols] = df[age_cols].multiply(scaling_factors, axis=1)
        print(df[age_cols].sum().sum())
        error = 1
        if error < tol:
            break
    return df


def mix_age_distribution(df, age_cols, total_col, reference_distribution, threshold=150):
    cell_count = 0
    for index, row in df.iterrows():
        pop = row[total_col]
        if threshold > pop > 0:
            cell_count += 1
            weight = pop / threshold
            adjusted_values = weight * row[age_cols] + (1 - weight) * reference_distribution * pop
            df.loc[index, age_cols] = adjusted_values  #/ adjusted_values.sum() * pop
    print(f"Adjusted {cell_count} cells.")
    print(f"Total cells: {len(df)}")
    return df


# Load preprocessed data
#one_km_df = pd.read_csv("3adjusted_1km_with_higher.csv")
#hundred_m_df = pd.read_csv("3adjusted_100m_with_higher.csv")
ten_km_df = pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.csv")

# Adjust the 10km level using IPF and mix age distribution
print("Adjusting age distribution at the 10km level...")
age_cols_10km_10er = [
    # 10er-Gruppen
    'Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a80undaelter_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter']

age_cols_10km_5er = [
    # 5er-Gruppen
    'Unter18_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a18bis29_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a30bis49_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a50bis64_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a65undaelter_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'
]
# Count nans
nans = ten_km_df[age_cols_10km_10er+age_cols_10km_5er].isna().sum().sum()
print(f"Total nans: {nans}")
# Set nans in given cols to 0
ten_km_df[age_cols_10km_10er+age_cols_10km_5er] = ten_km_df[age_cols_10km_10er+age_cols_10km_5er].fillna(0)
print("Nans set to 0.")

reference_distribution_10km_10er = ten_km_df[age_cols_10km_10er].sum() / ten_km_df[age_cols_10km_10er].sum().sum()
reference_distribution_10km_5er = ten_km_df[age_cols_10km_5er].sum() / ten_km_df[age_cols_10km_5er].sum().sum()
col_totals_10er = ten_km_df[age_cols_10km_10er].sum().copy()
col_totals_5er = ten_km_df[age_cols_10km_5er].sum().copy()
row_totals_10er = ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].copy()
row_totals_5er = ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].copy()
print(f"Diff in totals (should be 0): {row_totals_10er.sum() - row_totals_5er.sum()}")
ten_km_df = mix_age_distribution(ten_km_df, age_cols_10km_10er,
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
                                 reference_distribution_10km_10er)
ten_km_df = mix_age_distribution(ten_km_df, age_cols_10km_5er,
                                 "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter",
                                 reference_distribution_10km_5er)
ten_km_df = ipf_adjustment(ten_km_df, row_totals_10er, col_totals_10er, age_cols_10km_10er)
ten_km_df = ipf_adjustment(ten_km_df, row_totals_5er, col_totals_5er, age_cols_10km_5er)
ten_km_df.to_csv("3_4adjusted_10km_ipf.csv", index=False)

# Do NOT integerize
# Save
print("Saved adjusted 10km IPF results to 3_4adjusted_10km_ipf.csv")
print("Col totals differ a tiny bit between 10er and 5er groups, but that's ok and left that way so we keep the best possible distribution."
      "The total population is the same for both groups, and the row totals match exactly.")


Adjusting age distribution at the 10km level...
Total nans: 208
Nans set to 0.
Diff in totals (should be 0: 0.0
Adjusted 52 cells.
Total cells: 3824
Adjusted 52 cells.
Total cells: 3824
82711150.0
82711150.0
82711150.0
82711150.0
82711150.0
82711150.0
82711150.0
82711150.0
82711150.0
82711150.0
82710877.0
82710877.0
82710877.0
82710877.0
82710877.0
82710877.0
82710877.0
82710877.0
82710877.0
82710877.0
Saved adjusted 10km IPF results to 3_4adjusted_10km_ipf.csv


In [8]:
# Check the results
import pandas as pd
import numpy as np

adj_df = pd.read_csv("3_4adjusted_10km_ipf.csv")
orig_df = pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.csv")

diff_df = adj_df[age_cols_10km_10er+age_cols_10km_5er] - orig_df[age_cols_10km_10er+age_cols_10km_5er]
print(diff_df.abs().max())

# Zeige jeweils das maximum der Col an
print(orig_df[age_cols_10km_10er+age_cols_10km_5er].max())

# Print the diffs between row totals (should be 0) the diffs between col totals (can be a slight diff) and the diff betw row and col totals for each grouping
print(f"Row total diff 10er/5er: {adj_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].sum() - adj_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].sum()}")
print(f"Col total diff 10er/5er: {adj_df[age_cols_10km_10er].sum().sum() - adj_df[age_cols_10km_5er].sum().sum()}")
print(f"Row v Col total diff 10er: {adj_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].sum() - adj_df[age_cols_10km_10er].sum().sum()}")
print(f"Row v Col total diff 5er: {adj_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].sum() - adj_df[age_cols_10km_5er].sum().sum()}")


Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter         3.566970
a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        4.596714
a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        6.695852
a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        2.985058
a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        4.232525
a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        7.174324
a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        8.726717
a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter        6.543100
a80undaelter_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter    3.583652
Unter18_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter            4.886568
a18bis29_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter           8.151290
a30bis49_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter           6.287109
a50bis64_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter           7.563742
a65undaelter

In [1]:
# We now have good age distros for all 10km cells.
# Now, for 1km and 100km, mix age distributions for low pop cells downward from higher level, then do IPF

import pandas as pd
import numpy as np
import re
from tqdm import tqdm


def smart_integerize(df, pop_cols):
    for pop_col in pop_cols:
        total_before = df[pop_col].sum()
        df[pop_col] = df[pop_col].round().astype(int)
        total_after = df[pop_col].sum()
        diff = total_before - total_after
        if diff != 0:
            sorted_indices = df[pop_col].abs().sort_values(ascending=False).index[:abs(int(diff))]
            df.loc[sorted_indices, pop_col] += int(diff / abs(diff))
    return df


def ipf_adjustment(df, row_control_totals, col_control_totals, relevant_cols, max_iterations=10, tol=1e-6):
    for _ in range(max_iterations):

    # Here we do rows last, so the population total per cell is hit exactly

        col_totals = df[relevant_cols].sum()
        scaling_factors = col_control_totals / col_totals
        # Set inf to 1
        scaling_factors[~np.isfinite(scaling_factors)] = 1
        df[relevant_cols] = df[relevant_cols].multiply(scaling_factors, axis=1)
    
        row_totals = df[relevant_cols].sum(axis=1)
        scaling_factors = row_control_totals / row_totals
        # Set inf to 1
        scaling_factors[~np.isfinite(scaling_factors)] = 1
        df[relevant_cols] = df[relevant_cols].multiply(scaling_factors, axis=0)

        error = 1
        if error < tol:
            break
    return df


def mix_distributions(df, age_cols, total_col, reference_cols, threshold=100):
    mask = (df[total_col] > 0) & (df[total_col] < threshold)
    cell_count = mask.sum()

    if cell_count == 0:
        print("No cells adjusted.")
        return df

    weight = df.loc[mask, total_col] / threshold
    weight = weight.values.reshape(-1, 1)

    pop = df.loc[mask, total_col].values.reshape(-1, 1)
    age_data = df.loc[mask, age_cols].values
    # Fit age data
    age_data = age_data / age_data.sum(axis=1).reshape(-1, 1) * pop  # scale age data to pop
    #age_data = np.nan_to_num(age_data)
    ref_data = df.loc[mask, reference_cols].values * pop  # scale reference by pop

    adjusted = weight * age_data + (1 - weight) * ref_data
    df.loc[mask, age_cols] = adjusted

    adjusted_row_sums = df.loc[mask, age_cols].sum(axis=1)
    print(f"Row sums mixing: {adjusted_row_sums.sum()}")
    print(f"Total pop: {df.loc[mask, total_col].sum()}")
    #assert np.allclose(adjusted_row_sums, df.loc[mask, total_col], atol=1e-3), "Row sums do not match total pop."
    
    print(f"Adjusted {cell_count} cells.")
    print(f"Total cells: {len(df)}")
    return df


# Load preprocessed data
one_km_df = pd.read_csv("3adjusted_1km_with_higher.csv")
hundred_m_df = pd.read_csv("3adjusted_100m_with_higher.csv")
ten_km_df = pd.read_csv("3_4adjusted_10km_ipf.csv")

age_cols_10km_10er = [
    # 10er-Gruppen
    'Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter',
    'a80undaelter_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter']

age_cols_10km_5er = [
    # 5er-Gruppen
    'Unter18_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a18bis29_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a30bis49_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a50bis64_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter',
    'a65undaelter_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'
]

# 1km level
print("Adjusting age distribution at the 1km level...")

age_cols_1km_5er = [col.replace('10km', '1km') for col in age_cols_10km_5er]
age_cols_1km_10er = [col.replace('10km', '1km') for col in age_cols_10km_10er]
# Count nans
nans = one_km_df[age_cols_1km_10er+age_cols_1km_5er].isna().sum().sum()
print(f"Total nans: {nans}")
# Set nans in given cols to 0.00001 (to avoid div by 0)
one_km_df[age_cols_1km_10er+age_cols_1km_5er] = one_km_df[age_cols_1km_10er+age_cols_1km_5er].fillna(0.00001)
print("Nans set to 0.00001.")

ten_km_df['age_sum_10er'] = ten_km_df[age_cols_10km_10er].sum(axis=1) # Minimal!! anders als die "Insgesamt" Reihensumme (nur bei 10km!), deswegen so
ten_km_df['age_sum_5er'] = ten_km_df[age_cols_10km_5er].sum(axis=1)

for col in age_cols_10km_10er:
    ten_km_df[f'{col}_prop'] = ten_km_df[col] / ten_km_df['age_sum_10er']
for col in age_cols_10km_5er:
    ten_km_df[f'{col}_prop'] = ten_km_df[col] / ten_km_df['age_sum_5er']

age_prop_cols_10er = [f'{col}_prop' for col in age_cols_10km_10er]
age_prop_cols_5er = [f'{col}_prop' for col in age_cols_10km_5er]

one_km_df = one_km_df.merge(ten_km_df[['GITTER_ID_10km'] + age_prop_cols_5er + age_prop_cols_10er],
                            on='GITTER_ID_10km', how='left')

assert one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter'].sum() == one_km_df[
    'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'].sum()

# We now have the relative distributions of the 10km cells in the 1km cells. Now they are mixed where needed.
one_km_df = mix_distributions(one_km_df, age_cols_1km_10er,
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter', age_prop_cols_10er)
one_km_df = mix_distributions(one_km_df, age_cols_1km_5er,
                                    'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter', age_prop_cols_5er)

# Per-cell IPF: We have mixed distributions per 1km cell. To make sure the 1km cell distros also in sum match the 10km cells, AND
# the population totals per cell are still met (which they already are but adjusting distro messes with them) we do IPF, but per 10km cell.

grouped_1km = one_km_df.groupby('GITTER_ID_10km')
debug_counter = 0
for gid, group in tqdm(grouped_1km):
    debug_counter += 1
    # Only ipf 
    row_control_totals_10er = group['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter'].values.copy()
    # Get col totals from 10km cell
    col_control_totals_10er = ten_km_df.loc[ten_km_df['GITTER_ID_10km'] == gid, age_cols_10km_10er].values[0].copy()
    relevant_cols = age_cols_1km_10er
    group = ipf_adjustment(group, row_control_totals_10er, col_control_totals_10er, relevant_cols)
    one_km_df.loc[group.index, relevant_cols] = group[relevant_cols]

    row_control_totals_5er = group['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'].values[0].copy()
    # Get col totals from 10km cell
    col_control_totals_5er = ten_km_df.loc[ten_km_df['GITTER_ID_10km'] == gid, age_cols_10km_5er].values[0].copy()
    relevant_cols = age_cols_1km_5er
    group = ipf_adjustment(group, row_control_totals_5er, col_control_totals_5er, relevant_cols)
    one_km_df.loc[group.index, relevant_cols] = group[relevant_cols]
    if debug_counter > 20:
        break
    
# As result, we have the robust distros from more aggregated cells, but localized data mixed in. 
# NO integerization

# Save only top 100 rows
#one_km_df.head(100).to_csv("4adjusted_1km_mixing_ipf.csv", index=False)

one_km_df.to_csv("4adjusted_1km_mixing_ipf.csv", index=False)
print("Saved adjusted 1km IPF results to 4adjusted_1km_mixing_ipf.csv")

# Short check that the totals were not messed up (we don't edit them here but just to be sure)
print("Totals (should match):")
print(one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter'].sum())
print(one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'].sum())
print(ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].sum())
print(ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].sum())

totals_diffs = (ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].values -
                one_km_df.groupby('GITTER_ID_10km')['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter'].sum().values)
print(max(totals_diffs))
totals_diffs = (ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].values -
                one_km_df.groupby('GITTER_ID_10km')['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'].sum().values)
print(max(totals_diffs))



# -------------------------------------------------------------------------------------------
# 100m level with integerization
print("Adjusting age distribution at the 100m level...")
age_cols_100m_5er = [
    # 5er-Gruppen
    'Unter18_Zensus2022_Alter_5Altersklassen_100m-Gitter',
    'a18bis29_Zensus2022_Alter_5Altersklassen_100m-Gitter',
    'a30bis49_Zensus2022_Alter_5Altersklassen_100m-Gitter',
    'a50bis64_Zensus2022_Alter_5Altersklassen_100m-Gitter',
    'a65undaelter_Zensus2022_Alter_5Altersklassen_100m-Gitter'
]
# age_cols_1km_5er = [
#     # 5er-Gruppen
#     'Unter18_Zensus2022_Alter_5Altersklassen_1km-Gitter',
#     'a18bis29_Zensus2022_Alter_5Altersklassen_1km-Gitter',
#     'a30bis49_Zensus2022_Alter_5Altersklassen_1km-Gitter',
#     'a50bis64_Zensus2022_Alter_5Altersklassen_1km-Gitter',
#     'a65undaelter_Zensus2022_Alter_5Altersklassen_1km-Gitter'
# ]
#age_cols_100m_5er = [col.replace('10km', '100m') for col in age_cols_10km_5er]
age_cols_100m_10er = [col.replace('10km', '100m') for col in age_cols_10km_10er]
# Count nans
nans = hundred_m_df[age_cols_100m_5er+age_cols_100m_10er].isna().sum().sum()
print(f"Total nans: {nans}")
# Set nans in given cols to 0
hundred_m_df[age_cols_100m_5er+age_cols_100m_10er] = hundred_m_df[age_cols_100m_5er+age_cols_100m_10er].fillna(0.00001)
print("Nans set to 0.00001.")

#TODO: IPF is broken thus the prop cols dont add up to 1 and make 100m bad too
for col in age_cols_1km_10er:
    one_km_df[f'{col}_prop'] = one_km_df[col] / one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter']
for col in age_cols_1km_5er:
    one_km_df[f'{col}_prop'] = one_km_df[col] / one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter']
age_prop_cols_10er = [f'{col}_prop' for col in age_cols_1km_10er]
age_prop_cols_5er = [f'{col}_prop' for col in age_cols_1km_5er]

hundred_m_df = hundred_m_df.merge(one_km_df[['GITTER_ID_1km'] + age_prop_cols_5er + age_prop_cols_10er],
                            on='GITTER_ID_1km', how='left')

assert hundred_m_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter'].sum() == hundred_m_df[
    'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_100m-Gitter'].sum()

# We now have the relative distributions of the 1km cells in the 100m cells. Now they are mixed where needed (which is almost always at this level)

#age_cols_100m_10er = [col.replace('10km', '1km') for col in age_cols_10km_10er]
#age_cols_1km_5er = [col.replace('10km', '1km') for col in age_cols_10km_5er]

hundred_m_df = mix_distributions(hundred_m_df, age_cols_100m_10er,
                                 'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter', age_prop_cols_10er)
hundred_m_df = mix_distributions(hundred_m_df, age_cols_100m_5er,
                                    'Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_100m-Gitter', age_prop_cols_5er)


# Per-cell IPF: We have mixed distributions per 1km cell. To make sure the 1km cell distros also in sum match the 10km cells, AND
# the population totals per cell are still met (which they already are but adjusting distro messes with them) we do IPF, but per 10km cell.

grouped_100m = hundred_m_df.groupby('GITTER_ID_1km')
debug_counter = 0
for gid, group in tqdm(grouped_100m):
    debug_counter += 1
    row_control_totals_10er = group['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter'].values[0].copy()
    # Get col totals from 1km cell
    col_control_totals_10er = one_km_df.loc[one_km_df['GITTER_ID_1km'] == gid, age_cols_1km_10er].values[0].copy()
    relevant_cols = age_cols_100m_10er
    group = ipf_adjustment(group, row_control_totals_10er, col_control_totals_10er, relevant_cols)
    hundred_m_df.loc[group.index, relevant_cols] = group[relevant_cols]

    row_control_totals_5er = group['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_100m-Gitter'].values[0].copy()
    # Get col totals from 1km cell
    col_control_totals_5er = one_km_df.loc[one_km_df['GITTER_ID_1km'] == gid, age_cols_1km_5er].values[0].copy()
    relevant_cols = age_cols_100m_5er
    group = ipf_adjustment(group, row_control_totals_5er, col_control_totals_5er, relevant_cols)
    hundred_m_df.loc[group.index, relevant_cols] = group[relevant_cols]
    if debug_counter > 20:
        break
    
# As result, we have the robust distros from more aggregated cells, but localized data mixed in. 
# # Integerize (not for now while debugging)
hundred_m_df.to_csv("4adjusted_100m_mixing_ipf.csv", index=False)
print("Saved adjusted 1km IPF results to 4adjusted_1km_mixing_ipf.csv")


C:\Users\petre\AppData\Local\Temp\ipykernel_1568\466555968.py:77: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  one_km_df = pd.read_csv("3adjusted_1km_with_higher.csv")
C:\Users\petre\AppData\Local\Temp\ipykernel_1568\466555968.py:78: DtypeWarning: Columns (27,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  hundred_m_df = pd.read_csv("3adjusted_100m_with_higher.csv")


Adjusting age distribution at the 1km level...
Total nans: 719488
Nans set to 0.00001.
Row sums mixing: 3676946.0
Total pop: 3676946
Adjusted 122315 cells.
Total cells: 212746
Row sums mixing: 3676946.0
Total pop: 3676946
Adjusted 122315 cells.
Total cells: 212746


  1%|          | 20/3824 [00:11<36:35,  1.73it/s]


Saved adjusted 1km IPF results to 4adjusted_1km_mixing_ipf.csv
Totals (should match):
82711381
82711381
82711382.0
82711382.0
18.0
18.0
Adjusting age distribution at the 100m level...
Total nans: 21842251
Nans set to 0.00001.
Row sums mixing: 62775848.737544045
Total pop: 62778168
Adjusted 2963028 cells.
Total cells: 3147936
Row sums mixing: 62728384.679754764
Total pop: 62778168
Adjusted 2963028 cells.
Total cells: 3147936


Error evaluating: thread_id: pid_1568_id_2080410978448
frame_id: 2080753941104
scope: FRAME
attrs: hundred_m_df
Traceback (most recent call last):
  File "C:\Users\petre\AppData\Local\JetBrains\PyCharm 2024.1.2\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_resolver.py", line 178, in _getPyDictionary
    attr = getattr(var, n)
           ^^^^^^^^^^^^^^^
  File "C:\Users\petre\AppData\Local\miniforge3\envs\MATSimPipeline\Lib\site-packages\pandas\core\frame.py", line 3978, in T
    return self.transpose()
           ^^^^^^^^^^^^^^^^
  File "C:\Users\petre\AppData\Local\miniforge3\envs\MATSimPipeline\Lib\site-packages\pandas\core\frame.py", line 3937, in transpose
    new_arr = self.values.T
              ^^^^^^^^^^^
  File "C:\Users\petre\AppData\Local\miniforge3\envs\MATSimPipeline\Lib\site-packages\pandas\core\frame.py", line 12664, in values
    return self._mgr.as_array()
           ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\petre\AppData\Local\miniforge3\envs\MATSimPipeline\Lib\si

Saved adjusted 1km IPF results to 4adjusted_1km_mixing_ipf.csv


In [ ]:
 # Check results
one_km_df = pd.read_csv("4adjusted_1km_mixing_ipf.csv")
ten_km_df = pd.read_csv("3_4adjusted_10km_ipf.csv")
hundred_m_df = pd.read_csv("4adjusted_100m_ipf.csv")

# Short check that the totals were not messed up (we don't edit them here but just to be sure)
print("Totals (should match):")
print(one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter'].sum())
print(one_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'].sum())
print(ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].sum())
print(ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].sum())

totals_diffs = (ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter'].values -
                one_km_df.groupby('GITTER_ID_10km')['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter'].sum().valus())
print(max(totals_diffs))
totals_diffs = (ten_km_df['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_10km-Gitter'].values -
                one_km_df.groupby('GITTER_ID_10km')['Insgesamt_Bevoelkerung_Zensus2022_Alter_in_5_Altersklassen_1km-Gitter'].sum().values)
print(max(totals_diffs))

# Compare total distros
distro_diffs = (ten_km_df[age_cols_10km_10er].sum() - one_km_df[age_cols_1km_10er].sum()).abs()
print(distro_diffs.max())
distro_diffs = (ten_km_df[age_cols_10km_5er].sum() - one_km_df[age_cols_1km_5er].sum()).abs()
print(distro_diffs.max())
distro_diffs = (one_km_df[age_cols_1km_10er].sum() - hundred_m_df[age_cols_100m_10er].sum()).abs()
print(distro_diffs.max())
distro_diffs = (one_km_df[age_cols_1km_5er].sum() - hundred_m_df[age_cols_100m_5er].sum()).abs()
print(distro_diffs.max())

# Compare per-cell distros




In [10]:
#----------------------------------------
# Separate from above steps
# ----------------------------------------
# Rename col to fit the name of the provided cells geopackage.
import pandas as pd
df= pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\synthesis\synthetic_population\2poptotal_adjusted_100m_gitter.csv")

# df["GITTER_ID_1km_short"] = df["GITTER_ID_1km"].str.replace(
#     r"CRS3035RES1000mN(\d{7})E(\d{7})",
#     lambda m: f"1kmN{int(m.group(1))//1000}E{int(m.group(2))//1000}",
#     regex=True
# )

# df["numberOfHouseholds"] = df["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter"]

df = df.drop(columns=["Unnamed: 0.1"])

df.to_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\synthesis\synthetic_population\2poptotal_adjusted_100m_gitter.csv", index=False)


C:\Users\petre\AppData\Local\Temp\ipykernel_15908\3791361207.py:6: DtypeWarning: Columns (28,35,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\synthesis\synthetic_population\2poptotal_adjusted_100m_gitter.csv")


-----------------------------------------------------------------------------------



In [3]:
import pandas as pd

perturbation_probs = {
    0:  {0: 1.00},
    1:  {-1: 0.65, +2: 0.30, +3: 0.05},
    2:  {-2: 0.40, +1: 0.50, +2: 0.10},
    3:  {0: 0.40, +1: 0.35, +2: 0.15, -3: 0.05, +3: 0.05},
    4:  {0: 0.35, -1: 0.20, +1: 0.20, +2: 0.10, +3: 0.05, -4: 0.05, +4: 0.05},
    5:  {0: 0.35, -1: 0.175, +1: 0.175, -2: 0.10, +2: 0.10, +3: 0.05, +4: 0.05},
    6:  {0: 0.35, -1: 0.175, +1: 0.175, -2: 0.075, +2: 0.075, -3: 0.05, +3: 0.05, +4: 0.05},
    7:  {0: 0.35, -1: 0.15, +1: 0.15, -2: 0.075, +2: 0.075, -3: 0.05, +3: 0.05, -4: 0.05, +4: 0.05},
}

In [ ]:
# Load merged census files, clean up and save as pkl
merged_100 = pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.csv")
merged_1km = pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.csv")
merged_10_km = pd.read_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.csv")

# merged_100 = merged_100.rename(
#     columns={
#         col: col.replace("Alter_5Altersklassen", "Alter_in_5_Altersklassen")
#         for col in merged_100.columns
#         if "Alter_5Altersklassen" in col
#     }
# )
# Save as .pkl
merged_100.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.pkl")
merged_1km.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.pkl")
merged_10_km.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.pkl")


In [1]:
import pandas as pd

# Read the pickled DataFrames
merged_100 = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.pkl")
merged_1km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.pkl")
merged_10_km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.pkl")

# Rematch with csv
# Write DataFrames to CSV
merged_100.to_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.csv", index=False)
merged_1km.to_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.csv", index=False)
merged_10_km.to_csv(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.csv", index=False)



In [2]:
import pandas as pd
import re

# Load 100m data
merged_100 = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.pkl")

# Function to extract and link to 1km GITTER_ID
def link_to_1km(gitter_id):
    match = re.search(r"N(\d+)E(\d+)", gitter_id)
    if match:
        n = (int(match.group(1)) // 1000) * 1000
        e = (int(match.group(2)) // 1000) * 1000
        return f"CRS3035RES1000mN{n}E{e}"
    return None

# Add Linked_1km column
merged_100["Linked_1km"] = merged_100["GITTER_ID_100m"].apply(link_to_1km)

# Save the result
merged_100.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter_with_1km.pkl")
merged_100.head()


,GITTER_ID_100m,Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,...,Deutschland_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,EU27_Land_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Sonstiges_Europa_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Sonstige_Welt_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Insgesamt_Bevoelkerung_Zensus2022_Staatsangehoerigkeit_100m-Gitter,Deutschland_Zensus2022_Staatsangehoerigkeit_100m-Gitter,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_100m-Gitter,Linked_1km
0,CRS3035RES100mN2689100E4337000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.0,3.0,NaN,NaN,NaN,4.0,NaN,3.0,CRS3035RES1000mN2689000E4337000
1,CRS3035RES100mN2689100E4341100,11.0,NaN,4.0,NaN,NaN,3.0,NaN,NaN,3.0,...,7.0,NaN,NaN,NaN,NaN,NaN,11.0,7.0,NaN,CRS3035RES1000mN2689000E4341000
2,CRS3035RES100mN2690800E4341200,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,CRS3035RES1000mN2690000E4341000
3,CRS3035RES100mN2691200E4341100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRS3035RES1000mN2691000E4341000
4,CRS3035RES100mN2691200E4341200,12.0,NaN,NaN,6.0,3.0,3.0,NaN,NaN,NaN,...,8.0,3.0,3.0,NaN,NaN,NaN,12.0,8.0,3.0,CRS3035RES1000mN2691000E4341000


In [3]:
import pandas as pd
import re

# Load 1km data
merged_1km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.pkl")

# Function to extract and link to 10km GITTER_ID
def link_to_10km(gitter_id):
    match = re.search(r"N(\d+)E(\d+)", gitter_id)
    if match:
        n = (int(match.group(1)) // 10000) * 10000
        e = (int(match.group(2)) // 10000) * 10000
        return f"CRS3035RES10000mN{n}E{e}"
    return None

# Add Linked_10km column
merged_1km["Linked_10km"] = merged_1km["GITTER_ID_1km"].apply(link_to_10km)

# Save the result
merged_1km.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter_with_10km.pkl")
merged_1km.head()


,GITTER_ID_1km,Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,...,Deutschland_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,EU27_Land_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Sonstiges_Europa_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Sonstige_Welt_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Insgesamt_Bevoelkerung_Zensus2022_Staatsangehoerigkeit_1km-Gitter,Deutschland_Zensus2022_Staatsangehoerigkeit_1km-Gitter,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_1km-Gitter,Linked_10km
0,CRS3035RES1000mN2689000E4337000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.0,3.0,NaN,NaN,NaN,4.0,NaN,3.0,CRS3035RES10000mN2680000E4330000
1,CRS3035RES1000mN2689000E4341000,11.0,NaN,4.0,NaN,NaN,3.0,NaN,NaN,3.0,...,7.0,NaN,NaN,NaN,NaN,NaN,11.0,7.0,NaN,CRS3035RES10000mN2680000E4340000
2,CRS3035RES1000mN2690000E4341000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,CRS3035RES10000mN2690000E4340000
3,CRS3035RES1000mN2691000E4340000,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN,CRS3035RES10000mN2690000E4340000
4,CRS3035RES1000mN2691000E4341000,22.0,NaN,4.0,7.0,3.0,4.0,NaN,5.0,NaN,...,18.0,3.0,3.0,NaN,NaN,NaN,22.0,18.0,3.0,CRS3035RES10000mN2690000E4340000


In [ ]:
import pandas as pd

import pandas as pd
def compare_population_totals_with_suffix(lower_df, higher_df, linked_id_col, id_col, lower_suffix, higher_suffix):
    import pandas as pd

    # Step 1: Identify lower-level population columns
    pop_cols_lower = [col for col in lower_df.columns if lower_suffix in col]

    # Step 2: Generate mapping from lower to corresponding higher-level column names
    pop_mapping = {col: col.replace(lower_suffix, higher_suffix) for col in pop_cols_lower}

    # Step 3: Convert lower-level population columns to numeric & log NaN counts
    print("🔽 Coerced to NaN in lower-level DataFrame:")
    for col in pop_cols_lower:
        original_na = lower_df[col].isna().sum()
        lower_df[col] = pd.to_numeric(lower_df[col], errors='coerce')
        new_na = lower_df[col].isna().sum()
        if new_na > original_na:
            print(f"  - {col}: {new_na - original_na} new NaNs")

    # Step 4: Group and sum lower-level data
    lower_agg = lower_df.groupby(linked_id_col)[pop_cols_lower].sum().rename(columns=pop_mapping).reset_index()

    # Step 5: Convert higher-level population columns to numeric & log NaN counts
    print("🔼 Coerced to NaN in higher-level DataFrame:")
    for col in pop_mapping.values():
        original_na = higher_df[col].isna().sum()
        higher_df[col] = pd.to_numeric(higher_df[col], errors='coerce')
        new_na = higher_df[col].isna().sum()
        if new_na > original_na:
            print(f"  - {col}: {new_na - original_na} new NaNs")

    # Step 6: Merge
    comparison = pd.merge(
        higher_df[[id_col] + list(pop_mapping.values())],
        lower_agg,
        left_on=id_col,
        right_on=linked_id_col,
        suffixes=('_high', '_low'),
        how='inner'
    )

    # Step 7: Compute differences
    for col in pop_mapping.values():
        comparison[f"{col}_abs_diff"] = comparison[f"{col}_low"] - comparison[f"{col}_high"]
        comparison[f"{col}_rel_diff"] = (
            comparison[f"{col}_abs_diff"] / comparison[f"{col}_high"].replace(0, pd.NA)
        )

        # Step 8: Print summary statistics
    print("\n📊 Comparison Summary:")
    for col in pop_mapping.values():
        abs_diff_col = f"{col}_abs_diff"
        rel_diff_col = f"{col}_rel_diff"

        abs_diff = comparison[abs_diff_col]
        rel_diff = comparison[rel_diff_col]

        num_total = len(comparison)
        num_exact = (abs_diff == 0).sum()
        num_large_diff = (rel_diff.abs() > 0.01).sum()  # >1% relative error

        print(f"\n🔹 {col}")
        print(f"  - Total comparisons: {num_total}")
        print(f"  - Total diff:s:    {abs_diff.sum()}")
        print(f"  - Exact matches:     {num_exact} ({num_exact / num_total:.2%})")
        print(f"  - >1% difference:    {num_large_diff} ({num_large_diff / num_total:.2%})")
        print(f"  - Mean rel diff:     {rel_diff.mean():.4f}")
        print(f"  - Max rel diff:      {rel_diff.max():.4f}")
        print(f"  - Min rel diff:      {rel_diff.min():.4f}")

    return comparison



comparison_100_to_1km = compare_population_totals_with_suffix(
    lower_df=merged_100,
    higher_df=merged_1km,
    linked_id_col='Linked_1km',
    id_col='GITTER_ID_1km',
    lower_suffix='100m-Gitter',
    higher_suffix='1km-Gitter'
)

comparison_1km_to_10km = compare_population_totals_with_suffix(
    lower_df=merged_1km,
    higher_df=merged_10_km,
    linked_id_col='Linked_10km',
    id_col='GITTER_ID_10km',
    lower_suffix='1km-Gitter',
    higher_suffix='10km-Gitter'
)

In [ ]:
import numpy as np
import math
import pandas as pd

# Redefine get_likelihood and perturbation_probs after reset

perturbation_probs = {
    0: {0: 1.00},
    1: {-1: 0.65, 2: 0.30, 3: 0.05},
    2: {-2: 0.40, 1: 0.50, 2: 0.10},
    3: {0: 0.40, 1: 0.35, 2: 0.15, -3: 0.05, 3: 0.05},
    4: {0: 0.35, -1: 0.20, 1: 0.20, 2: 0.10, 3: 0.05, -4: 0.05, 4: 0.05},
    5: {0: 0.35, -1: 0.175, 1: 0.175, -2: 0.10, 2: 0.10, 3: 0.05, 4: 0.05},
    6: {0: 0.35, -1: 0.175, 1: 0.175, -2: 0.075, 2: 0.075, -3: 0.05, 3: 0.05, 4: 0.05},
    7: {0: 0.35, -1: 0.15, 1: 0.15, -2: 0.075, 2: 0.075, -3: 0.05, 3: 0.05, -4: 0.05, 4: 0.05},
}

def get_likelihood(observed, true):
    base_true = true if true in perturbation_probs else 7
    distribution = perturbation_probs.get(base_true, perturbation_probs[7])
    delta = observed - true
    return distribution.get(delta, 0.0)

def greedy_bayesian_adjust_multistep(observed_vals, target_sum, max_step=4):
    n = len(observed_vals)

    best_estimates = []
    for obs in observed_vals:
        candidates = range(max(0, obs - max_step), obs + max_step + 1)
        best = max(candidates, key=lambda t: get_likelihood(obs, t))
        best_estimates.append(best)

    current_sum = sum(best_estimates)
    delta = target_sum - current_sum

    if delta == 0:
        return best_estimates

    adjustments = []
    for i, (obs, est) in enumerate(zip(observed_vals, best_estimates)):
        for step in range(1, max_step + 1):
            if delta > 0 and est + step <= obs + max_step:
                new_likelihood = get_likelihood(obs, est + step)
                old_likelihood = get_likelihood(obs, est)
                if new_likelihood > 0 and old_likelihood > 0:
                    loss = -np.log(new_likelihood) + np.log(old_likelihood)
                    adjustments.append((loss, i, step))
            elif delta < 0 and est - step >= 0:
                new_likelihood = get_likelihood(obs, est - step)
                old_likelihood = get_likelihood(obs, est)
                if new_likelihood > 0 and old_likelihood > 0:
                    loss = -np.log(new_likelihood) + np.log(old_likelihood)
                    adjustments.append((loss, i, -step))

    # Sort adjustments by smallest loss
    adjustments.sort()

    # Apply adjustments greedily
    remaining = abs(delta)
    for _, i, step in adjustments:
        if remaining <= 0:
            break
        best_estimates[i] += step
        remaining -= abs(step)

    return best_estimates



# Load data
merged_100 = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.pkl")
merged_1km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.pkl")
merged_10km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_gitter.pkl")


# 10km -> 1km adjustment
runs=0
limiter = 10
for gid_10km in merged_10km["GITTER_ID_10km"].unique():
    runs+=1
    if runs%1 == 0:
        print(runs)
    if runs > limiter:
        print("Limit reached")
        break
    target = merged_10km.loc[merged_10km["GITTER_ID_10km"] == gid_10km, "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter"].values
    if len(target) == 0: continue
    target = int(round(target[0]))
    idx_1km = merged_1km["Linked_10km"] == gid_10km
    observed = merged_1km.loc[idx_1km, "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter"]
    observed = observed.fillna(0)
    observed = observed.astype(int).tolist()
    if len(observed) == 0: continue
    corrected = greedy_bayesian_adjust_multistep(observed, target)
    merged_1km.loc[idx_1km, f"Insgesamt_Bevoelkerung"] = corrected

# 1km -> 100m adjustment
runs=0
limiter = 10
for gid_1km in merged_1km["GITTER_ID_1km"].unique():
    runs+=1
    if runs%1 == 0:
        print(runs)
    if runs > limiter:
        print("Limit reached")
        break
    target = merged_1km.loc[merged_1km["GITTER_ID_1km"] == gid_1km, "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter"].values
    if len(target) == 0: continue
    target = int(round(target[0]))
    idx_100m = merged_100["Linked_1km"] == gid_1km
    observed = merged_100.loc[idx_100m, "Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter"]
    observed = observed.fillna(0)
    observed = observed.astype(int).tolist()
    if len(observed) == 0: continue
    corrected = greedy_bayesian_adjust_multistep(observed, target)
    merged_100.loc[idx_100m, f"Insgesamt_Bevoelkerung"] = corrected

merged_100["Bevoelkerung_adjust_diff"] = merged_100["Insgesamt_Bevoelkerung"] - merged_100["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter"]
merged_1km["Bevoelkerung_adjust_diff"] = merged_1km["Insgesamt_Bevoelkerung"] - merged_1km["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter"]
print (f"Summe 100m:{merged_100["Insgesamt_Bevoelkerung"].sum()}")
print (f"Summe 1km:{merged_1km["Insgesamt_Bevoelkerung"].sum()}")
print (f"Summe 10km:{merged_10km["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter"].sum()}")
# # Save results
# merged_1km.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_adjusted.pkl")
# merged_100.to_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_adjusted.pkl")


In [2]:
merged_100.head(100)

,GITTER_ID_100m,Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter,...,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,EU27_Land_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Sonstiges_Europa_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Sonstige_Welt_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_100m-Gitter,Insgesamt_Bevoelkerung_Zensus2022_Staatsangehoerigkeit_100m-Gitter,Deutschland_Zensus2022_Staatsangehoerigkeit_100m-Gitter,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_100m-Gitter,Linked_1km,Insgesamt_Bevoelkerung
0,CRS3035RES100mN2689100E4337000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,NaN,NaN,NaN,4.0,NaN,3.0,CRS3035RES1000mN2689000E4337000,4.0
1,CRS3035RES100mN2689100E4341100,11.0,NaN,4.0,NaN,NaN,3.0,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,11.0,7.0,NaN,CRS3035RES1000mN2689000E4341000,11.0
2,CRS3035RES100mN2690800E4341200,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,CRS3035RES1000mN2690000E4341000,4.0
3,CRS3035RES100mN2691200E4341100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRS3035RES1000mN2691000E4341000,0.0
4,CRS3035RES100mN2691200E4341200,12.0,NaN,NaN,6.0,3.0,3.0,NaN,NaN,NaN,...,3.0,3.0,NaN,NaN,NaN,12.0,8.0,3.0,CRS3035RES1000mN2691000E4341000,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CRS3035RES100mN2698700E4338400,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN,CRS3035RES1000mN2698000E4338000,NaN
96,CRS3035RES100mN2698700E4339400,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN,CRS3035RES1000mN2698000E4339000,NaN
97,CRS3035RES100mN2698700E4341800,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.0,7.0,NaN,CRS3035RES1000mN2698000E4341000,NaN
98,CRS3035RES100mN2698700E4341900,5.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,5.0,5.0,NaN,CRS3035RES1000mN2698000E4341000,NaN


In [3]:
merged_1km.head(100)

,GITTER_ID_1km,Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter,...,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,EU27_Land_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Sonstiges_Europa_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Sonstige_Welt_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Sonstige_Zensus2022_Staatsangehoerigkeit_Gruppen_1km-Gitter,Insgesamt_Bevoelkerung_Zensus2022_Staatsangehoerigkeit_1km-Gitter,Deutschland_Zensus2022_Staatsangehoerigkeit_1km-Gitter,Ausland_Sonstige_Zensus2022_Staatsangehoerigkeit_1km-Gitter,Linked_10km,Insgesamt_Bevoelkerung
0,CRS3035RES1000mN2689000E4337000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,NaN,NaN,NaN,4.0,NaN,3.0,CRS3035RES10000mN2680000E4330000,4.0
1,CRS3035RES1000mN2689000E4341000,11.0,NaN,4.0,NaN,NaN,3.0,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,11.0,7.0,NaN,CRS3035RES10000mN2680000E4340000,8.0
2,CRS3035RES1000mN2690000E4341000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,CRS3035RES10000mN2690000E4340000,6.0
3,CRS3035RES1000mN2691000E4340000,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN,CRS3035RES10000mN2690000E4340000,3.0
4,CRS3035RES1000mN2691000E4341000,22.0,NaN,4.0,7.0,3.0,4.0,NaN,5.0,NaN,...,3.0,3.0,NaN,NaN,NaN,22.0,18.0,3.0,CRS3035RES10000mN2690000E4340000,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CRS3035RES1000mN2705000E4341000,977.0,105.0,73.0,95.0,123.0,107.0,134.0,133.0,115.0,...,149.0,63.0,50.0,30.0,NaN,977.0,830.0,149.0,CRS3035RES10000mN2700000E4340000,977.0
96,CRS3035RES1000mN2705000E4342000,258.0,21.0,13.0,27.0,25.0,30.0,51.0,44.0,28.0,...,32.0,30.0,5.0,NaN,NaN,258.0,229.0,32.0,CRS3035RES10000mN2700000E4340000,258.0
97,CRS3035RES1000mN2705000E4343000,296.0,27.0,27.0,38.0,31.0,33.0,50.0,50.0,24.0,...,18.0,4.0,10.0,5.0,NaN,296.0,273.0,18.0,CRS3035RES10000mN2700000E4340000,296.0
98,CRS3035RES1000mN2705000E4395000,23.0,NaN,NaN,NaN,NaN,7.0,4.0,NaN,NaN,...,20.0,13.0,NaN,NaN,NaN,23.0,NaN,20.0,CRS3035RES10000mN2700000E4390000,23.0


In [8]:
import pandas as pd

merged_100 = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_gitter.pkl")
merged_1km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_gitter.pkl")
merged_100_adj= pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_poptotaladjusted.pkl")
merged_1km_adj= pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_poptotaladjusted.pkl")
# Sum 100m results grouped to 1km
sum_100m_by_1km = merged_100.groupby("Linked_1km")["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_100m-Gitter"].sum()
# Get 1km adjusted targets
in1km = merged_1km.set_index("GITTER_ID_1km")["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_1km-Gitter"]

sum_100m_by_1km_adj = merged_100_adj.groupby("Linked_1km")["Insgesamt_Bevoelkerung"].sum()
in1km_adj = merged_1km_adj.set_index("GITTER_ID_1km")["Insgesamt_Bevoelkerung"]


# Compute mismatch
mismatch = (sum_100m_by_1km - in1km)
print (len(mismatch))
mismatch = mismatch.dropna()
print (len(mismatch))
mismatch = mismatch[mismatch != 0]
print (mismatch)

mismatch = (sum_100m_by_1km_adj - in1km_adj)
print (len(mismatch))
mismatch = mismatch.dropna()
print (len(mismatch))
mismatch = mismatch[mismatch != 0]
print (mismatch)

212789
209812
CRS3035RES1000mN2691000E4341000     1.0
CRS3035RES1000mN2692000E4344000    -2.0
CRS3035RES1000mN2693000E4340000     2.0
CRS3035RES1000mN2694000E4340000    -2.0
CRS3035RES1000mN2694000E4343000    -5.0
                                   ... 
CRS3035RES1000mN3545000E4219000   -13.0
CRS3035RES1000mN3546000E4219000    -8.0
CRS3035RES1000mN3546000E4220000   -13.0
CRS3035RES1000mN3547000E4219000    -5.0
CRS3035RES1000mN3547000E4220000     4.0
Length: 172726, dtype: float64
212789
211430
Series([], Name: Insgesamt_Bevoelkerung, dtype: float64)


In [9]:
missing_1km_ids = set(merged_1km["GITTER_ID_1km"]) - set(merged_100["Linked_1km"])
print(f"{len(missing_1km_ids)} 1km cells missing from 100m data")
missing_1km_ids

1316 1km cells missing from 100m data


{'CRS3035RES1000mN3278000E4278000',
 'CRS3035RES1000mN3296000E4387000',
 'CRS3035RES1000mN3339000E4449000',
 'CRS3035RES1000mN2841000E4515000',
 'CRS3035RES1000mN3005000E4117000',
 'CRS3035RES1000mN3045000E4133000',
 'CRS3035RES1000mN3156000E4662000',
 'CRS3035RES1000mN2819000E4328000',
 'CRS3035RES1000mN3391000E4561000',
 'CRS3035RES1000mN3512000E4249000',
 'CRS3035RES1000mN3267000E4595000',
 'CRS3035RES1000mN3293000E4151000',
 'CRS3035RES1000mN2730000E4376000',
 'CRS3035RES1000mN3158000E4403000',
 'CRS3035RES1000mN2992000E4085000',
 'CRS3035RES1000mN3317000E4445000',
 'CRS3035RES1000mN3268000E4160000',
 'CRS3035RES1000mN3341000E4568000',
 'CRS3035RES1000mN3268000E4386000',
 'CRS3035RES1000mN3275000E4322000',
 'CRS3035RES1000mN3329000E4567000',
 'CRS3035RES1000mN3315000E4350000',
 'CRS3035RES1000mN2944000E4420000',
 'CRS3035RES1000mN3303000E4363000',
 'CRS3035RES1000mN2733000E4376000',
 'CRS3035RES1000mN3155000E4439000',
 'CRS3035RES1000mN2789000E4512000',
 'CRS3035RES1000mN3532000E42

In [14]:
# Total number of persons in these 1km cells (pre-adjustment)
missing_persons = merged_1km_adj.loc[
    merged_1km_adj["GITTER_ID_1km"].isin(missing_1km_ids),
    "Insgesamt_Bevoelkerung"
].sum()

print(f"Total population in 1km cells missing from 100m data: {missing_persons}")

Total population in 1km cells missing from 100m data: 3079.0


In [2]:
import pandas as pd
import numpy as np
from ortools.linear_solver import pywraplp
from tqdm import tqdm
import cvxpy as cp

# Load data
merged_10km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_poptotaladjusted.pkl")
merged_1km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_poptotaladjusted.pkl")
merged_100m = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_poptotaladjusted.pkl")

# Age columns
age_cols_10y = [
    "Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a80undaelter_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter"
]

age_cols_100m = [c.format("100m") for c in age_cols_10y]
age_cols_1km = [c.format("1km") for c in age_cols_10y]
age_cols_10km = [c.format("10km") for c in age_cols_10y]

# Global national distribution
global_dist = np.array([
    7_758_664, 7_580_167, 9_158_337,
    10_855_510, 10_000_497, 12_926_437,
    11_059_795, 7_338_960, 6_041_173
], dtype=np.float64) # added 2 on last value bcs these are missing in the official total
global_dist /= global_dist.sum()

def normalize(v):
    v = np.array(v, dtype=np.float64)
    s = v.sum()
    return v / s if s > 0 else np.zeros_like(v)

def blend(n_high, n_local, total_local, threshold=200):
    p_high = normalize(n_high)
    p_local = normalize(n_local)
    alpha = max(0.0, min(1.0, (threshold - total_local) / threshold))
    if alpha < 0.001:
        return n_local
    return (alpha * p_high + (1 - alpha) * p_local) * total_local


def ipf_ndarray(seed_matrix, target_rows, target_cols, max_iter=1000, tol=1e-5):
    A = seed_matrix.astype(float).copy()

    for _ in range(max_iter):
        # Scale rows
        row_sums = A.sum(axis=1)
        row_factors = np.divide(target_rows, row_sums, out=np.ones_like(row_sums), where=row_sums != 0)
        A = (A.T * row_factors).T

        # Scale columns
        col_sums = A.sum(axis=0)
        col_factors = np.divide(target_cols, col_sums, out=np.ones_like(col_sums), where=col_sums != 0)
        A *= col_factors

        # Check convergence
        if (np.allclose(A.sum(axis=1), target_rows, atol=tol) and
            np.allclose(A.sum(axis=0), target_cols, atol=tol)):
            print("Converged")
            print(f"Row sums: {A.sum(axis=1)}")
            print(f"Target row:{target_rows}")
            print(f"Col sums: {A.sum(axis=0)}")
            print(f"Target col:{target_cols}")
            break
    return A


def min_cost_integer_matrix_qp(blended, row_sums, col_sums):
    print("Blended shape (9, N):", blended.shape)  # should be (9, N)
    print("Row sums shape (9,):", row_sums.shape)  # (9,)
    print("Col sums shape (N,):", col_sums.shape)  # (N,)
    total_rows = np.sum(row_sums)
    total_cols = np.sum(col_sums)
    assert blended.shape[0] == len(row_sums), f"Row mismatch: {blended.shape[0]} rows vs {len(row_sums)} totals"
    assert blended.shape[1] == len(col_sums), f"Column mismatch: {blended.shape[1]} cols vs {len(col_sums)} totals"
    assert np.isclose(np.sum(row_sums), np.sum(col_sums)), "Total mismatch!"
    assert np.isclose(total_rows, total_cols), f"Row sum = {total_rows}, col sum = {total_cols}"
    assert np.all(row_sums >= 0)
    assert np.all(col_sums >= 0)

    A, C = blended.shape
    X = cp.Variable((A, C), integer=True)

    # Objective: minimize squared deviation from blended
    objective = cp.Minimize(cp.sum_squares(X - blended))

    # Constraints: row sums
    constraints = [
        cp.sum(X, axis=1) == row_sums,
        cp.sum(X, axis=0) == col_sums,
        X >= 0
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.ECOS_BB, verbose = True)

    if prob.status != cp.OPTIMAL:
        raise RuntimeError("No optimal integer solution found.")

    return np.rint(X.value).astype(int)


print("Replacing nans")
# Replace all NaNs with 0 in relevant age columns
merged_10km[age_cols_10km] = merged_10km[age_cols_10km].fillna(0)
# merged_10km["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter"] = merged_10km["Insgesamt_Bevoelkerung_Zensus2022_Alter_in_10er-Jahresgruppen_10km-Gitter"].fillna(0) # just 2 empty ones, no problem in the smaller ones (we handled that before)
merged_1km[age_cols_1km] = merged_1km[age_cols_1km].fillna(0)
merged_100m[age_cols_100m] = merged_100m[age_cols_100m].fillna(0)


print("🔄 Blending 10km cells with national distribution and integerizing...")

blended_10km_prior = []
cell_totals_10km = []
for _, row in tqdm(merged_10km.iterrows(), total=len(merged_10km)):
    raw = row[age_cols_10km].values
    total = row["Insgesamt_Bevoelkerung"]
    blended = blend(global_dist, raw, total)
    blended_10km_prior.append(blended)
    cell_totals_10km.append(total)

# Construct matrix for OR
blended_10km_prior = np.array(blended_10km_prior).T  # shape (age_groups, cells)
cell_totals_10km = np.array(cell_totals_10km)
national_age_totals_calc = global_dist * cell_totals_10km.sum()
national_age_totals = np.array([
    7_758_664,  # Unter 10
    7_580_167,  # 10–19
    9_158_337,  # 20–29
    10_855_510, # 30–39
    10_000_497, # 40–49
    12_926_437, # 50–59
    11_059_795, # 60–69
    7_338_960,  # 70–79
    6_041_173   # 80+ # added 2 on last value bcs these are missing in the official total
], dtype=int)
assert np.allclose(national_age_totals_calc, national_age_totals, atol=0.5), \
    "⚠️ Mismatch between calculated and given national age group totals!"


################
# blended_10km_prior: (num_cells, 9) — per-cell distributions
# Convert to correct matrix shape: (9, num_cells)
blended_full = np.array(blended_10km_prior)  # shape: (9, num_cells)
col_sums = np.array(cell_totals_10km, dtype=int)  # total population per cell

# 🔍 Subsample to 10 cells
blended_sub = blended_full[:, :10]      # shape (9, 10)
col_sums_sub = col_sums[:10]             # shape (10,)
# 📐 Compute adjusted row sums (age group totals) for just these 10 cells
sub_total = col_sums_sub.sum()
row_sums_sub = np.rint(global_dist * sub_total).astype(int)

# 🩹 Fix rounding mismatch
delta = row_sums_sub.sum() - sub_total
if delta != 0:
    row_sums_sub[np.argmax(row_sums_sub)] -= delta

# ✅ Sanity check
assert np.sum(row_sums_sub) == np.sum(col_sums_sub), "Totals don't match"
assert blended_sub.shape == (9, 10), "Shape mismatch for solver"

blended_sub_ipf = ipf_ndarray(blended_sub, row_sums_sub, col_sums_sub)
blended_sub_ipf_int = np.rint(blended_sub_ipf).astype(int)

# 🚀 Solve small instance
result_test = min_cost_integer_matrix_qp(blended_sub_ipf_int, row_sums_sub, col_sums_sub)

# 🔍 Confirm result
print("✅ Result shape:", result_test.shape)
print("Row sums:", result_test.sum(axis=1))
print("Col sums:", result_test.sum(axis=0))

######################

# Integerize via OR
blended_10km_vals = min_cost_integer_matrix_qp(blended_10km_prior * cell_totals_10km, national_age_totals, cell_totals_10km)

# Write results to DataFrame
blended_10km_vals = np.array(blended_10km_vals, dtype=int).T  # transpose back
for i, col in enumerate(age_cols_10km):
    merged_10km_blended[col.replace("10km", "adj")] = blended_10km_vals[:, i]

merged_10km_blended.to_pickle("merged_10km_agedistadjusted.pkl")
merged_10km_blended.to_csv("merged_10km_agedistadjusted.csv", index=False)

# 1km blend using adjusted 10km
print("🔄 Blending 1km cells using adjusted 10km marginals...")
merged_1km_blended = merged_1km.copy()
for gid_10km, group in tqdm(merged_1km.groupby("Linked_10km"), total=merged_1km["Linked_10km"].nunique()):
    if gid_10km not in merged_10km_blended["GITTER_ID_10km"].values:
        continue
    age_totals = merged_10km_blended.set_index("GITTER_ID_10km").loc[gid_10km, [c.replace("10km", "adj") for c in age_cols_10km]].values
    total_pops = group["Insgesamt_Bevoelkerung"].values
    blended_prior = np.array([
        blend(age_totals, row[age_cols_1km].values, total)
        for _, row, total in zip(group.index, group.itertuples(), total_pops)
    ])
    mat = blended_prior.T * total_pops
    integerized = min_cost_integer_matrix_qp(mat, age_totals, total_pops)
    for j, irow in enumerate(group.index):
        for k, col in enumerate(age_cols_1km):
            merged_1km_blended.at[irow, col.replace("1km", "adj")] = integerized[k, j]

# 100m blend using adjusted 1km
print("🔄 Blending 100m cells using adjusted 1km marginals...")
merged_100m_blended = merged_100m.copy()
for gid_1km, group in tqdm(merged_100m.groupby("Linked_1km"), total=merged_100m["Linked_1km"].nunique()):
    if gid_1km not in merged_1km_blended["GITTER_ID_1km"].values:
        continue
    age_totals = merged_1km_blended.set_index("GITTER_ID_1km").loc[gid_1km, [c.replace("1km", "adj") for c in age_cols_1km]].values
    total_pops = group["Insgesamt_Bevoelkerung"].values
    blended_prior = np.array([
        blend(age_totals, row[age_cols_100m].values, total)
        for _, row, total in zip(group.index, group.itertuples(), total_pops)
    ])
    mat = blended_prior.T * total_pops
    integerized = min_cost_integer_matrix_qp(mat, age_totals, total_pops)
    for j, irow in enumerate(group.index):
        for k, col in enumerate(age_cols_100m):
            merged_100m_blended.at[irow, col.replace("100m", "adj")] = integerized[k, j]

print("✅ All levels blended and integerized successfully.")

# --- Export final adjusted DataFrames
merged_1km_blended.to_pickle("merged_1km_agedistadjusted.pkl")
merged_100m_blended.to_pickle("merged_100m_agedistadjusted.pkl")

merged_1km_blended.to_csv("merged_1km_agedistadjusted.csv", index=False)
merged_100m_blended.to_csv("merged_100m_agedistadjusted.csv", index=False)

Replacing nans
🔄 Blending 10km cells with national distribution and integerizing...


100%|██████████| 3824/3824 [00:01<00:00, 1937.27it/s]

Converged
Row sums: [2041.99969236 1994.99985407 2409.99992617 2856.9997164  2631.99980923
 3401.00013999 2910.0001665  1931.00036797 1590.0003273 ]
Target row:[2042 1995 2410 2857 2632 3401 2910 1931 1590]
Col sums: [6.000e+00 1.100e+01 3.190e+02 6.072e+03 2.800e+01 2.200e+02 2.821e+03
 8.531e+03 7.950e+02 2.965e+03]
Target col:[   6   11  319 6072   28  220 2821 8531  795 2965]
Blended shape (9, N): (9, 10)
Row sums shape (9,): (9,)
Col sums shape (N,): (10,)
                                     CVXPY                                     
                                     v1.6.5                                    
(CVXPY) May 11 04:55:09 PM: Your problem has 90 variables, 109 constraints, and 0 parameters.
(CVXPY) May 11 04:55:09 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 11 04:55:09 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) May 11 04:55:09 PM: CVXPY will first compile your problem

-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
(CVXPY) May 11 04:55:09 PM: Invoking solver ECOS_BB  to obtain a solution.
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
(CVXPY) May 11 04:55:17 PM: Problem status: infeasible_inaccurate
(CVXPY) May 11 04:55:17 PM: Optimal value: inf
(CVXPY) May 11 04:55:17 PM: Compilation took 2.684e-02 seconds
(CVXPY) May 11 04:55:17 PM: Solver (including time spent in interface) took 8.731e+00 seconds


C:\Users\petre\AppData\Local\miniforge3\envs\MATSimPipeline\Lib\site-packages\cvxpy\problems\problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


RuntimeError: No optimal integer solution found.

In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- Load data ---
merged_10km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_10km_poptotaladjusted.pkl")
merged_1km = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_1km_poptotaladjusted.pkl")
merged_100m = pd.read_pickle(r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\syn_pop\merged_100m_poptotaladjusted.pkl")

merged_10km = merged_10km.fillna(0)
merged_1km = merged_1km.fillna(0)
merged_100m = merged_100m.fillna(0)

# --- Age columns ---
age_cols_template = [
    "Unter10_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a10bis19_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a20bis29_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a30bis39_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a40bis49_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a50bis59_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a60bis69_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a70bis79_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter",
    "a80undaelter_Zensus2022_Alter_in_10er-Jahresgruppen_{}-Gitter"
]
age_cols_10km = [c.format("10km") for c in age_cols_template]
age_cols_1km = [c.format("1km") for c in age_cols_template]
age_cols_100m = [c.format("100m") for c in age_cols_template]

# --- Global national age distribution ---
national_age_totals = np.array([
    7_758_664, 7_580_167, 9_158_337, 10_855_510, 10_000_497,
    12_926_437, 11_059_795, 7_338_960, 6_041_173
], dtype=np.float64)
global_dist = national_age_totals / national_age_totals.sum()

# --- Helper functions ---
def normalize(v):
    v = np.array(v, dtype=np.float64)
    s = v.sum()
    return v / s if s > 0 else np.zeros_like(v)


def blend(n_high, n_local, total_local, threshold=200):
    p_high = normalize(n_high)
    p_local = normalize(n_local)
    alpha = max(0.0, min(1.0, (threshold - total_local) / threshold))
    if alpha < 0.001:
        return n_local
    return (alpha * p_high + (1 - alpha) * p_local) * total_local

def ipf_single(blend_mat, row_targets, col_targets, max_iter=1000, tol=1e-6):
    A = blend_mat.astype(float).copy()
    row_targets = np.array(row_targets, dtype=np.float64)
    col_targets = np.array(col_targets, dtype=np.float64)
    for _ in range(max_iter):
        # Scale rows
        row_sums = A.sum(axis=1)
        row_factors = np.divide(row_targets, row_sums, out=np.ones_like(row_sums), where=row_sums != 0)
        A = (A.T * row_factors).T

        # Scale columns
        col_sums = A.sum(axis=0)
        col_factors = np.divide(col_targets, col_sums, out=np.ones_like(col_sums), where=col_sums != 0)
        A *= col_factors

        # Check convergence
        if (np.allclose(A.sum(axis=1), row_targets, atol=tol) and
            np.allclose(A.sum(axis=0), col_targets, atol=tol)):
            # print("Converged")
            # print(f"Row sums: {A.sum(axis=1)}")
            # print(f"Target row:{row_targets}")
            # print(f"Col sums: {A.sum(axis=0)}")
            # print(f"Target col:{col_targets}")
            # max_diff = np.max(np.abs(A - blend_mat))
            # print("Max absolute difference:", max_diff)
            break
    return A

# --- Process 10km ---
print("🔄 Adjusting 10km...")
merged_10km_blended = merged_10km.copy()
totals_10km = merged_10km["Insgesamt_Bevoelkerung"].values
raw_10km = merged_10km[age_cols_10km].fillna(0).values.T
blended_10km = np.array([blend(global_dist, raw_10km[:, i], totals_10km[i]) for i in range(raw_10km.shape[1])]).T
adjusted_10km = ipf_single(blended_10km, national_age_totals, totals_10km)
for i, col in enumerate(age_cols_10km):
    merged_10km_blended[col.replace("10km", "adj")] = adjusted_10km[i]
    adj_col = col.replace("10km", "adj")
    diff_col = col.replace("10km", "diff")
    merged_10km_blended[diff_col] = merged_10km_blended[adj_col] - merged_10km_blended[col]


# --- Process 1km ---
print("🔄 Adjusting 1km...")
merged_1km_blended = merged_1km.copy()
for gid_10km, group in tqdm(merged_1km.groupby("Linked_10km")):
    if gid_10km not in merged_10km_blended["GITTER_ID_10km"].values:
        continue
    age_targets = merged_10km_blended.set_index("GITTER_ID_10km").loc[gid_10km, [c.replace("10km", "adj") for c in age_cols_10km]].values
    group_total = group["Insgesamt_Bevoelkerung"].fillna(0).values
    raw = group[age_cols_1km].fillna(0).values.T
    blended = np.array([blend(age_targets, raw[:, i], group_total[i]) for i in range(raw.shape[1])]).T
    adjusted = ipf_single(blended, age_targets, group_total)
    for j, idx in enumerate(group.index):
        for i, col in enumerate(age_cols_1km):
            merged_1km_blended.at[idx, col.replace("1km", "adj")] = adjusted[i, j]
for i, col in enumerate(age_cols_1km):
    adj_col = col.replace("1km", "adj")
    diff_col = col.replace("1km", "diff")
    merged_1km_blended[diff_col] = merged_1km_blended[adj_col] - merged_1km_blended[col]


# --- Process 100m ---
print("🔄 Adjusting 100m...")
print("🔄 Precomputing 1km targets...")
# Set index once for fast, safe lookup
_1km_indexed = merged_1km_blended.set_index("GITTER_ID_1km")
# Precompute age targets as a dictionary: GITTER_ID_1km → target vector
age_targets_1km = {
    gid: _1km_indexed.loc[gid, [c.replace("1km", "adj") for c in age_cols_1km]].values
    for gid in tqdm(_1km_indexed.index, desc="⏳ Building target lookup")
}

merged_100m_blended = merged_100m.copy()
for gid_1km, group in tqdm(merged_100m.groupby("Linked_1km")):
    if gid_1km not in merged_1km_blended["GITTER_ID_1km"].values:
        continue
    # age_targets = merged_1km_blended.set_index("GITTER_ID_1km").loc[gid_1km, [c.replace("1km", "adj") for c in age_cols_1km]].values
    age_targets = age_targets_1km.get(gid_1km)
    if age_targets is None:
        continue
    group_total = group["Insgesamt_Bevoelkerung"].fillna(0).values
    raw = group[age_cols_100m].fillna(0).values.T
    blended = np.array([blend(age_targets, raw[:, i], group_total[i]) for i in range(raw.shape[1])]).T
    adjusted = ipf_single(blended, age_targets, group_total)
    for j, idx in enumerate(group.index):
        for i, col in enumerate(age_cols_100m):
            merged_100m_blended.at[idx, col.replace("100m", "adj")] = adjusted[i, j]
for i, col in enumerate(age_cols_100m):
    adj_col = col.replace("100m", "adj")
    diff_col = col.replace("100m", "diff")
    merged_100m_blended[diff_col] = merged_100m_blended[adj_col] - merged_100m_blended[col]


# --- Save results ---
merged_10km_blended.to_pickle("merged_10km_agedistadjusted_ipf.pkl")
merged_1km_blended.to_pickle("merged_1km_agedistadjusted_ipf.pkl")
merged_100m_blended.to_pickle("merged_100m_agedistadjusted_ipf.pkl")

merged_10km_blended.to_csv("merged_10km_agedistadjusted_ipf.csv", index=False)
merged_1km_blended.to_csv("merged_1km_agedistadjusted_ipf.csv", index=False)
merged_100m_blended.to_csv("merged_100m_agedistadjusted_ipf.csv", index=False)

print("✅ All levels blended, IPF-adjusted, and saved.")


🔄 Adjusting 10km...
🔄 Adjusting 1km...


100%|██████████| 3824/3824 [01:21<00:00, 47.11it/s]


🔄 Adjusting 100m...
🔄 Precomputing 1km targets...


100%|██████████| 211473/211473 [40:49<00:00, 86.33it/s]  


✅ All levels blended, IPF-adjusted, and saved.


In [ ]:
# This script filters census cells to those with buildings and assigns census cell IDs to buildings (ALKIS)

import geopandas as gpd

# --- File paths ---
alkis_path = r"T:\petre\UCFL\Synthetic Population\ALKIS Buildings\gebaeude-ni.shp"
census_path = r"T:\petre\UCFL\Synthetic Population\Zensus\DE_Grid_ETRS89-LAEA_100m.gpkg"
# census_layer = "your_layer_name"  # replace with actual layer name
output_filtered_cells = "filtered_census_cells.gpkg"
output_enriched_buildings = "buildings_with_cell_ids.gpkg"

# --- Load data ---
print("Loading data...")
buildings = gpd.read_file(alkis_path)
buildings.crs = "EPSG:25832"  # Set the CRS for buildings if not already set
census = gpd.read_file(census_path)

# --- Ensure same CRS ---
if buildings.crs != census.crs:
    print(f"Census crs:{census.crs}, buildings crs: {buildings.crs}")
    print("Reprojecting census data to match buildings CRS...")
    census = census.to_crs(buildings.crs)
# --- Filter census cells intersecting buildings ---
print("Filtering census cells that contain buildings...")
census_filtered = census[census.intersects(buildings.unary_union)]

# --- Add formatted 100m cell ID from x_sw and y_sw ---
def format_ids_from_sw_coords(gdf, x_col="x_sw", y_col="y_sw"):
    x = gdf[x_col].astype(int)
    y = gdf[y_col].astype(int)
    return np.char.add(
        np.char.add("CRS3035RES100mN", y.astype(str)),
        np.char.add("E", x.astype(str))
    )

census_filtered["formatted_cell_id"] = format_ids_from_sw_coords(census_filtered)

# --- Use building centroids for join ---
print("Assigning census cell ID to each building (centroid-based)...")
buildings_centroids = buildings.copy()
buildings_centroids["geometry"] = buildings_centroids.centroid

# Add unique building ID to retain linkage
buildings_centroids["__building_idx__"] = buildings_centroids.index

# Spatial join to assign cell ID based on centroid location
buildings_with_ids = gpd.sjoin(
    buildings_centroids[["__building_idx__", "geometry"]],
    census_filtered[[census_filtered.geometry.name, "formatted_cell_id"]],
    how="left",
    predicate="within"
)
print("Validating")
# --- Validation ---
# Should be one row per building
grouped = buildings_with_ids.groupby("__building_idx__").size()
duplicates = grouped[grouped > 1]
missing = grouped[grouped == 0]

assert duplicates.empty, f"Buildings with multiple matches found: {duplicates}"
assert len(missing) == 0, f"Some buildings were not assigned a cell: {len(missing)}"

# Join result back to original buildings
matched_ids = buildings_with_ids.set_index("__building_idx__")["formatted_cell_id"]
buildings["cell_id"] = matched_ids

assert buildings["cell_id"].notna().all(), "Some buildings have missing cell_id values."


# --- Save results ---
print("Saving GeoDataFrames as pickles...")

buildings.to_pickle("buildings_with_cell_ids.pkl")
census_filtered.to_pickle("filtered_census_cells.pkl")

print("Pickle files saved.")

print("Saving filtered census cells...")
census_filtered.to_file(output_filtered_cells, driver="GPKG")

print("Saving buildings with assigned cell IDs...")
buildings.to_file(output_enriched_buildings, driver="GPKG")

print("Done.")


Loading data...
Census crs:EPSG:3035, buildings crs: EPSG:25832
Reprojecting census data to match buildings CRS...
Filtering census cells that contain buildings...


C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_15652\3198392517.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  census_filtered = census[census.intersects(buildings.unary_union)]


In [3]:
import geopandas as gpd
import numpy as np

# --- File paths ---
alkis_path = r"C:\Users\bienzeisler\Desktop\ALKIS Buildings\gebaeude-ni.shp"
filtered_census_path = r"C:\Users\bienzeisler\Documents\NI100mGitter.gpkg"
output_enriched_buildings = "buildings_with_cell_ids.gpkg"

# --- Load data ---
print("Loading buidlings")
buildings = gpd.read_file(alkis_path)
buildings.crs = "EPSG:25832"  # Ensure correct CRS
print("Loading filtered census cells")
census_filtered = gpd.read_file(filtered_census_path)
print("Census crs:", census_filtered.crs)
print("Buildings crs:", buildings.crs)
print("Reprojecting census data to match buildings CRS...")
census_filtered = census_filtered.to_crs(buildings.crs)  # Ensure same CRS

# --- Format cell ID from x_sw and y_sw ---
def format_ids_from_sw_coords(gdf, x_col="x_sw", y_col="y_sw"):
    x = gdf[x_col].astype(int)
    y = gdf[y_col].astype(int)
    return np.char.add(
        np.char.add("CRS3035RES100mN", y.astype(str)),
        np.char.add("E", x.astype(str))
    )

if "formatted_cell_id" not in census_filtered.columns:
    print("Generating formatted cell IDs...")
    census_filtered["formatted_cell_id"] = format_ids_from_sw_coords(census_filtered)

# --- Assign census cell ID to each building using centroid ---
print("Assigning census cell IDs to buildings (centroid-based)...")
buildings_centroids = buildings.copy()
buildings_centroids["geometry"] = buildings_centroids.centroid
buildings_centroids["__building_idx__"] = buildings_centroids.index

# Perform spatial join
buildings_with_ids = gpd.sjoin(
    buildings_centroids[["__building_idx__", "geometry"]],
    census_filtered[[census_filtered.geometry.name, "formatted_cell_id"]],
    how="left",
    predicate="within"
)
print("Validating...")
# Validation
grouped = buildings_with_ids.groupby("__building_idx__").size()
duplicates = grouped[grouped > 1]
missing = buildings_centroids.index.difference(buildings_with_ids["__building_idx__"])

assert duplicates.empty, f"Buildings with multiple matches found: {duplicates}"
assert len(missing) == 0, f"Buildings not assigned to a cell: {len(missing)}"

# Reattach to original buildings GeoDataFrame
matched_ids = buildings_with_ids.set_index("__building_idx__")["formatted_cell_id"]
buildings["cell_id"] = matched_ids

if buildings["cell_id"].notna().all():
    print("Some buildings have missing cell_id values.")
    print("Missing cell IDs:", buildings[buildings["cell_id"].isna()])

# --- Save enriched buildings ---
print("Saving buildings with cell IDs...")
buildings.to_file(output_enriched_buildings, driver="GPKG")
print("Done.")


Loading buidlings
Loading filtered census cells
Census crs: EPSG:3035
Buildings crs: EPSG:25832
Reprojecting census data to match buildings CRS...
Generating formatted cell IDs...
Assigning census cell IDs to buildings (centroid-based)...
Validating...
Saving buildings with cell IDs...
Done.


In [9]:
import geopandas as gpd
import pandas as pd
import numpy as np
print("Loading")
# --- Load the GeoPackage ---
gdf = gpd.read_file(r"C:\Users\bienzeisler\Documents\NI100mGitter.gpkg")

# --- Format cell ID from x_sw and y_sw ---
def format_ids_from_sw_coords(gdf, x_col="x_sw", y_col="y_sw"):
    x = gdf[x_col].astype(int)
    y = gdf[y_col].astype(int)
    return np.char.add(
        np.char.add("CRS3035RES100mN", y.astype(str)),
        np.char.add("E", x.astype(str))
    )
print("Generating formatted cell IDs...")
# --- Create new column with formatted ID ---
gdf["long_id"] = format_ids_from_sw_coords(gdf)

print("Creating output DataFrame...")
# --- Create output DataFrame ---
df = pd.DataFrame()
df["ZENSUS100m"] = gdf["long_id"]
df["STAAT"] = 1
df["WELT"] = 1
print("Saving to CSV...")
# --- Save to CSV ---
df.to_csv("geocross_NI.csv", index=False)
print("Saving to GeoPackage...")
# --- Save updated GeoPackage ---
gdf.to_file("NI100mGitter_with_long_id.gpkg", driver="GPKG")


Loading
Generating formatted cell IDs...
Creating output DataFrame...
Saving to CSV...
Saving to GeoPackage...


In [12]:
# This splits a large cvs into 10 parts:
# 1% of the data in the first file and 15 equal parts of the remaining data in the other files.

import pandas as pd

# --- Load the full DataFrame ---
df = pd.read_csv("geocross_NI.csv")  # or use df directly

# --- Define sizes ---
total_len = len(df)
one_percent_size = total_len // 100
remaining = total_len - one_percent_size
batch_size = remaining // 15

# --- Write the 1% batch ---
df.iloc[:one_percent_size].to_csv("geocross_NI_00.csv", index=False)

# --- Write the remaining 15 batches ---
for i in range(15):
    start = one_percent_size + i * batch_size
    # Last batch takes the remainder
    end = one_percent_size + (i + 1) * batch_size if i < 14 else total_len
    df.iloc[start:end].to_csv(f"geocross_NI_{i+1:02d}.csv", index=False)

print("✅ Custom batching complete.")


✅ Custom batching complete.


In [10]:
import fiona

with fiona.open(r"NI100mGitter_with_long_id.gpkg") as layer:
    # Column names
    print("Columns:", list(layer.schema["properties"].keys()))

    # First row (as dict)
    first_feature = next(iter(layer))
    print("First row:", first_feature["properties"])


Columns: ['rowid', 'featuretype_name', 'dataset_name', 'OBJECTID', 'id', 'x_sw', 'y_sw', 'x_mp', 'y_mp', 'f_staat', 'f_land', 'f_wasser', 'p_staat', 'p_land', 'p_wasser', 'Shape_Length', 'Shape_Area', 'ags', 'id_2', 'Objektidentifikator', 'BeginnLebenszeit', 'Admin_Ebene_ADE', 'ADE', 'Geofaktor_GF', 'GF', 'Besondere_Gebiete_BSG', 'BSG', 'Regionalschlüssel_ARS', 'Gemeindeschlüssel_AGS', 'Verwaltungssitz_SDV_ARS', 'GeografischerName_GEN', 'Bezeichnung', 'Identifikator_IBZ', 'Bemerkung', 'Namensbildung_NBD', 'NBD', 'Land', 'Regierungsbezirk', 'Kreis', 'VerwaltungsgemeinschaftTeil1', 'VerwaltungsgemeinschaftTeil2', 'Gemeinde', 'Funk_Schlüsselstelle3', 'FK_S3', 'Europ_Statistikschlüssel_NUTS', 'RegioschlüsselAufgefüllt', 'GemeindeschlüsselAufgefüllt', 'Wirksamkeit_WSK', 'long_id']
First row: fiona.Properties(rowid=18711700, featuretype_name='DE_Grid_ETRS89-LAEA_100m', dataset_name='de_grid_laea_100m', OBJECTID=18711700, id='100mN31326E42998', x_sw=4299800.0, y_sw=3132600.0, x_mp=4299850.0, 

In [15]:
import shutil
import os

# --- Paths ---
base_folder = r"T:\petre\UCFL\Synthetic Population"
source_folder = os.path.join(base_folder, "4_Niedersachsen_00")
csv_source_folder = os.path.join(base_folder, "geocross_parts")  # Adjust if needed
csv_target_name = "_prep0_geo_cross_walk.csv"

# --- Create 15 copies ---
for i in range(1, 16):
    suffix = f"{i:02d}"
    target_folder = os.path.join(base_folder, f"4_Niedersachsen_{suffix}")
    
    # Copy the entire folder
    shutil.copytree(src=source_folder, dst=target_folder)
    
    # Define source and target for the CSV
    original_csv = f"geocross_NI_{suffix}.csv"
    target_csv = os.path.join(target_folder, "data", csv_target_name)

    # Copy and rename CSV
    shutil.copy2(original_csv, target_csv)

    print(f"✅ {target_folder}: inserted as /data/{csv_target_name}")

print("🎯 All folders created and crosswalks renamed.")


✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_01: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_02: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_03: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_04: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_05: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_06: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_07: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_08: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_09: inserted as /data/_prep0_geo_cross_walk.csv
✅ T:\petre\UCFL\Synthetic Population\4_Niedersachsen_10: inserted as /data/_prep0_geo_cross

In [5]:
import zipfile
from pathlib import Path

# === INPUTS ===
zip_path = Path(r"C:\Users\petre\Downloads\popsim_NI_results.zip")  # <-- change this
output_dir = Path(r"C:\Users\petre\Downloads\extracted_household_ids")
output_dir.mkdir(exist_ok=True)

# === Open ZIP archive ===
with zipfile.ZipFile(zip_path, "r") as z:
    matching_files = [f for f in z.namelist() if f.endswith("output/final_expanded_household_ids.csv")]
    print(f"📦 Gefundene Dateien: {len(matching_files)}")

    for file in matching_files:
        # Use folder name as output file name
        parts = Path(file).parts
        folder_name = parts[-3]  # e.g. .../<folder>/output/...
        dest_file = output_dir / f"{folder_name}_households.csv"

        print(f"🔽 Extrahiere {file} → {dest_file.name}")
        with z.open(file) as src, open(dest_file, "wb") as dst:
            dst.write(src.read())

print(f"\n✅ Fertig. Dateien gespeichert in: {output_dir.resolve()}")


📦 Gefundene Dateien: 58
🔽 Extrahiere fertig_00/output/final_expanded_household_ids.csv → fertig_00_households.csv
🔽 Extrahiere fertig_14/output/final_expanded_household_ids.csv → fertig_14_households.csv
🔽 Extrahiere fertig_old_4_Niedersachsen_01_a/output/final_expanded_household_ids.csv → fertig_old_4_Niedersachsen_01_a_households.csv
🔽 Extrahiere fertig_old_4_Niedersachsen_01_b/output/final_expanded_household_ids.csv → fertig_old_4_Niedersachsen_01_b_households.csv
🔽 Extrahiere fertig_old_4_Niedersachsen_01_c/output/final_expanded_household_ids.csv → fertig_old_4_Niedersachsen_01_c_households.csv
🔽 Extrahiere fertig_old_4_Niedersachsen_01_d/output/final_expanded_household_ids.csv → fertig_old_4_Niedersachsen_01_d_households.csv
🔽 Extrahiere old_4_Niedersachsen_02_a/output/final_expanded_household_ids.csv → old_4_Niedersachsen_02_a_households.csv
🔽 Extrahiere old_4_Niedersachsen_02_b/output/final_expanded_household_ids.csv → old_4_Niedersachsen_02_b_households.csv
🔽 Extrahiere old_4_N

In [7]:
import pandas as pd
from pathlib import Path

input_dir = Path(r"C:\Users\petre\Downloads\extracted_household_ids")
output_file = Path(r"C:\Users\petre\Downloads\merged_households.csv")

# === Load and concatenate all CSVs ===
print(f"📂 Lade CSV-Dateien aus {input_dir}...")
csv_files = sorted(input_dir.glob("*.csv"))

dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)
    print(f"  ✅ {file.name}: {len(df)} Zeilen")

merged = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Gesamtzeilen nach Merge: {len(merged):,}")

# === Check for long_id duplicates ===
if "ZENSUS100m" not in merged.columns:
    raise ValueError("❌ Spalte 'long_id' nicht gefunden!")

duplicates = merged["ZENSUS100m"].duplicated(keep=False)
num_duplicates = duplicates.sum()

print(f"🔍 Doppelte long_id-Einträge: {num_duplicates:,}")

# (Optional) Show example duplicates
if num_duplicates > 0:
    print("\n🔁 Beispielhafte Duplikate:")
    print(merged[duplicates].head())

# === Save merged CSV ===
merged.to_csv(output_file, index=False)
print(f"\n💾 Ergebnis gespeichert unter: {output_file.resolve()}")


📂 Lade CSV-Dateien aus C:\Users\petre\Downloads\extracted_household_ids...
  ✅ fertig_00_households.csv: 22566 Zeilen
  ✅ fertig_14_households.csv: 115000 Zeilen
  ✅ fertig_old_4_Niedersachsen_01_a_households.csv: 73815 Zeilen
  ✅ fertig_old_4_Niedersachsen_01_b_households.csv: 51341 Zeilen
  ✅ fertig_old_4_Niedersachsen_01_c_households.csv: 43313 Zeilen
  ✅ fertig_old_4_Niedersachsen_01_d_households.csv: 42454 Zeilen
  ✅ old_4_Niedersachsen_02_a_households.csv: 77223 Zeilen
  ✅ old_4_Niedersachsen_02_b_households.csv: 119095 Zeilen
  ✅ old_4_Niedersachsen_02_c_households.csv: 65791 Zeilen
  ✅ old_4_Niedersachsen_02_d_households.csv: 66446 Zeilen
  ✅ old_4_Niedersachsen_03_a_households.csv: 58885 Zeilen
  ✅ old_4_Niedersachsen_03_b_households.csv: 31342 Zeilen
  ✅ old_4_Niedersachsen_03_c_households.csv: 37833 Zeilen
  ✅ old_4_Niedersachsen_03_d_households.csv: 36276 Zeilen
  ✅ old_4_Niedersachsen_04_a_households.csv: 65971 Zeilen
  ✅ old_4_Niedersachsen_04_b_households.csv: 81966 Zeil

In [3]:
import pandas as pd

# === File paths ===
file1 = r"T:\petre\UCFL\Synthetic Population\geocross_NI.csv"
file2 = r"T:\petre\UCFL\Synthetic Population\merged_NI_households.csv"

# === Load data ===
print("📄 Lade geocross_NI.csv...")
df1 = pd.read_csv(file1)
print("📄 Lade merged_NI_households.csv...")
df2 = pd.read_csv(file2)

# === Ensure column exists
assert "ZENSUS100m" in df1.columns, "❌ 'ZENSUS100m' fehlt in geocross_NI.csv"
assert "ZENSUS100m" in df2.columns, "❌ 'ZENSUS100m' fehlt in merged_NI_households.csv"

# === Basic stats
print(f"\n🔢 Zeilenanzahl:")
print(f"  geocross_NI:               {len(df1):,}")
print(f"  merged_NI_households:      {len(df2):,}")

print(f"\n🔍 Eindeutige ZENSUS100m:")
u1 = set(df1["ZENSUS100m"].dropna().unique())
u2 = set(df2["ZENSUS100m"].dropna().unique())
print(f"  geocross_NI:               {len(u1):,}")
print(f"  merged_NI_households:      {len(u2):,}")

# === Compare sets
missing_in_merged = u1 - u2
missing_in_geocross = u2 - u1

print(f"\n❓ In geocross_NI, aber nicht in merged:    {len(missing_in_merged):,}")
print(f"❓ In merged, aber nicht in geocross_NI:    {len(missing_in_geocross):,}")

# Optional: show a few
if missing_in_merged:
    print("\n🧪 Beispiel fehlend in merged_NI:")
    print(list(missing_in_merged)[:10])

if missing_in_geocross:
    print("\n🧪 Beispiel fehlend in geocross_NI:")
    print(list(missing_in_geocross)[:10])


📄 Lade geocross_NI.csv...
📄 Lade merged_NI_households.csv...

🔢 Zeilenanzahl:
  geocross_NI:               4,778,609
  merged_NI_households:      3,690,999

🔍 Eindeutige ZENSUS100m:
  geocross_NI:               4,778,609
  merged_NI_households:      416,380

❓ In geocross_NI, aber nicht in merged:    4,362,229
❓ In merged, aber nicht in geocross_NI:    0

🧪 Beispiel fehlend in merged_NI:
['CRS3035RES100mN3351100E4283200', 'CRS3035RES100mN3207600E4291000', 'CRS3035RES100mN3167200E4315200', 'CRS3035RES100mN3246600E4376100', 'CRS3035RES100mN3188700E4295500', 'CRS3035RES100mN3232500E4289400', 'CRS3035RES100mN3361700E4190700', 'CRS3035RES100mN3376600E4132500', 'CRS3035RES100mN3207600E4281800', 'CRS3035RES100mN3298300E4219300']


In [1]:
import geopandas as gpd
import pandas as pd

# === Paths ===
gpkg_path = r"C:\Users\petre\Downloads\buildings_with_cell_ids.gpkg"
gpkg_layer = None  # use None if there's only one layer
csv_households = r"C:\Users\petre\Downloads\merged_households.csv"
csv_activities = r"T:\petre\UCFL\Synthetic Population\alkis_building_activity_map.csv"
#
# # === Load GeoPackage ===
# print("📦 Lade GeoPackage...")
# gdf = gpd.read_file(gpkg_path, layer=gpkg_layer)
# print(f"✅ Gebäude geladen: {len(gdf)} Zeilen")
#
# # === Load activities
# print("📄 Lade Aktivitätsdaten...")
# df_activities = pd.read_csv(csv_activities)
# if "gfk_code" not in df_activities.columns or "activities" not in df_activities.columns:
#     raise ValueError("❌ 'gfk_code' oder 'activities' fehlen in der Aktivitäten-CSV")
#
# # === Join activities to buildings
# print("🔗 Füge Aktivitäten zu Gebäuden hinzu...")
# gdf = gdf.merge(df_activities, how="left", left_on="GFK", right_on="gfk_code")
#
# # === Save after joining activities XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
# gdf.to_pickle("gdf_with_activities.pkl")
# print("💾 Zwischenspeicher gespeichert: gdf_with_activities.pkl")
# gdf = pd.read_pickle("gdf_with_activities.pkl")
#
# # === Mark 'home' buildings
# print("🏠 Erkenne Gebäude mit Aktivität 'home'...")
# gdf["has_home"] = gdf["activities"].fillna("").str.contains(r"\bhome\b", case=False)
# print(f"  ➤ Gebäude mit 'home': {gdf['has_home'].sum()} von {len(gdf)}")
#
# # === Load households CSV
# print("📄 Lade Haushaltsdaten...")
# df = pd.read_csv(csv_households)
# if "ZENSUS100m" not in df.columns or "H_ID" not in df.columns:
#     raise ValueError("❌ CSV muss 'ZENSUS100m' und 'H_ID' enthalten")
# print(f"Anzahl Haushalte davor:{len(df)}")
#
# from collections import defaultdict
# import numpy as np
#
# # === Build mapping: cell_id → buildings with home
# print("📐 Erzeuge Verteilungsmatrix pro Zelle...")
# cell_to_buildings = defaultdict(list)
# for row in gdf[gdf["has_home"]].itertuples(index=False):
#     cell_to_buildings[row.cell_id].append(row.OI)
#
# # === Build mapping: cell_id → households
# cell_to_households = df.groupby("ZENSUS100m", sort=False)["H_ID"].agg(list).to_dict()
#
# # === Build fallback: all buildings per cell (any use)
# print("📐 Erzeuge Fallback-Verzeichnis für Zellen ohne 'home'...")
# cell_to_all_buildings = defaultdict(list)
# for row in gdf.itertuples(index=False):
#     cell_to_all_buildings[row.cell_id].append(row.OI)
#
# # === Assign households to buildings
# print("🎯 Weise Haushalte Gebäuden zu...")
# household_assignment = defaultdict(list)
# cells_with_fallback = []
#
# for cell_id, hhs in cell_to_households.items():
#     building_idxs = cell_to_buildings.get(cell_id)
#
#     # Fallback if no home buildings
#     if not building_idxs:
#         building_idxs = cell_to_all_buildings.get(cell_id)
#         if not building_idxs:
#             print(f"⚠️  Keine Gebäude in Zelle {cell_id} – {len(hhs)} Haushalte werden übersprungen.")
#             continue
#         print(f"⚠️  Zelle {cell_id} hat keine 'home'-Gebäude – nutze {len(building_idxs)} beliebige Gebäude.")
#         cells_with_fallback.append(cell_id)
#
#     for i, hh in enumerate(hhs):
#         building_idx = building_idxs[i % len(building_idxs)]
#         household_assignment[building_idx].append(hh)
#
# print(f"✅ Fallback verwendet für {len(cells_with_fallback)} Zellen.")


# === Assign synthetic_households column XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
# gdf["synthetic_households"] = gdf["OI"].map(household_assignment.get)
# # === Save full building GDF with households
# gdf.to_pickle("gdf_with_households.pkl")
# print("💾 Zwischenspeicher gespeichert: gdf_with_households.pkl")
gdf = pd.read_pickle("gdf_with_households.pkl")

# === Summary
num_with_hh = gdf["synthetic_households"].notna().sum()
print(f"✅ Gebäude mit zugewiesenen Haushalten: {num_with_hh} / {len(gdf)}")
gdf["hh_count"] = gdf["synthetic_households"].apply(lambda x: len(x) if isinstance(x, list) else 0)
gdf["HH_IDs"] = gdf["synthetic_households"].apply(lambda x: ";".join(map(str, x)) if isinstance(x, list) else None)
gdf_with_hh = gdf.dropna(subset=["synthetic_households"]).copy()
gdf = gdf.drop(columns=["synthetic_households"])
gdf_with_hh.to_file("onlywithhh.gpkg", driver="GPKG")
print("Saved withhh only")
# === Save result
output_path = "buildings_with_households.gpkg"
gdf.to_file(output_path, driver="GPKG")
print(f"💾 Ergebnis gespeichert unter: {output_path}")
#
# # === Inverse mapping: household → building + centroid
# print("🔁 Erzeuge inverse Zuordnung Haushalte → Gebäude...")
# gdf_with_hh["centroid"] = gdf_with_hh.geometry.centroid
#
# inverse = gdf_with_hh.explode("synthetic_households", ignore_index=True)
# inverse = inverse.rename(columns={
#     "synthetic_households": "H_ID",
#     "cell_id": "assigned_cell_id",
#     "geometry": "building_geometry"
# })
#
# inverse_output = inverse[["H_ID", "assigned_cell_id", "centroid", "building_geometry"]]
# inverse_csv_path = "households_with_building_locations.csv"
# print(f"Anzahl Haushalte: {len(inverse_output)}")
# inverse_output.to_csv(inverse_csv_path, index=False)
# print(f"💾 Inverse CSV gespeichert unter: {inverse_csv_path}")


✅ Gebäude mit zugewiesenen Haushalten: 2210146 / 7582736
Saved withhh only
💾 Ergebnis gespeichert unter: buildings_with_households.gpkg


In [ ]:
inverse_output[inverse_output["assigned_cell_id"]=="CRS3035RES100mN3391300E4100400"]

In [2]:
import fiona
import geopandas as gpd

# === Path to file ===
path = r"T:\petre\UCFL\Synthetic Population\buildings_with_cell_ids.gpkg"  # <-- change this

# === Load only the metadata ===
print("📋 Lade nur Spaltennamen...")
with fiona.open(path) as f:
    print(f"🧾 Spalten: {list(f.schema['properties'].keys())}")
    print(f"🌍 Geometrie-Typ: {f.schema['geometry']}")

# === Load just the first row ===
print("\n🔍 Erste Zeile:")
gdf_sample = gpd.read_file(path, rows=1)  # requires GeoPandas 0.13+
print(gdf_sample.head(1).T)


📋 Lade nur Spaltennamen...
🧾 Spalten: ['AGS', 'OI', 'GFK', 'cell_id']
🌍 Geometrie-Typ: MultiPolygon

🔍 Erste Zeile:
                                                          0
AGS                                                03451004
OI                                         DENIAL1200007t8x
GFK                                              31001_1000
cell_id                      CRS3035RES100mN3336600E4185100
geometry  MULTIPOLYGON (((431105.415 5886206.835, 431110...


In [1]:
import pandas as pd

# === Load data ===
original_path = r"C:\Users\petre\Downloads\merged_households.csv"
inverse_path = "households_with_building_locations.csv"

print("📄 Lade Originaldaten...")
df_orig = pd.read_csv(original_path)
print("📄 Lade Inverse-Zuordnung...")
df_inv = pd.read_csv(inverse_path)

# === Group by ZENSUS100m / assigned_cell_id
print("\n🔄 Vergleiche Haushaltszuweisungen pro Zelle...")

orig_grouped = df_orig.groupby("ZENSUS100m")["H_ID"].apply(set)
inv_grouped = df_inv.groupby("assigned_cell_id")["H_ID"].apply(set)

# === Align index
all_cells = sorted(set(orig_grouped.index).union(inv_grouped.index))
orig_grouped = orig_grouped.reindex(all_cells, fill_value=set())
inv_grouped = inv_grouped.reindex(all_cells, fill_value=set())

# === Compare sets
mismatches = []
for cell in all_cells:
    if orig_grouped[cell] != inv_grouped[cell]:
        mismatches.append(cell)

# === Summary
print(f"\n✅ Zellen gesamt: {len(all_cells):,}")
print(f"✅ Übereinstimmungen: {len(all_cells) - len(mismatches):,}")
print(f"❌ Abweichungen: {len(mismatches):,} ({len(mismatches) / len(all_cells):.2%})")

# === Show sample mismatches
if mismatches:
    print("\n🔁 Beispielhafte Abweichungen:")
    for cell in mismatches[:10]:
        orig_set = orig_grouped[cell]
        inv_set = inv_grouped[cell]
        print(f"\n📍 Zelle: {cell}")
        print(f"  Original ({len(orig_set)}): {sorted(list(orig_set))[:5]} ...")
        print(f"  Inverse  ({len(inv_set)}): {sorted(list(inv_set))[:5]} ...")


📄 Lade Originaldaten...
📄 Lade Inverse-Zuordnung...

🔄 Vergleiche Haushaltszuweisungen pro Zelle...

✅ Zellen gesamt: 416,380
✅ Übereinstimmungen: 412,482
❌ Abweichungen: 3,898 (0.94%)

🔁 Beispielhafte Abweichungen:

📍 Zelle: CRS3035RES100mN3133100E4298900
  Original (1): [88266020] ...
  Inverse  (0): [] ...

📍 Zelle: CRS3035RES100mN3133100E4299000
  Original (5): [41435200, 59391500, 89440620, 89774440, 98831180] ...
  Inverse  (0): [] ...

📍 Zelle: CRS3035RES100mN3133200E4299100
  Original (2): [5540520, 20023970] ...
  Inverse  (0): [] ...

📍 Zelle: CRS3035RES100mN3133500E4298500
  Original (4): [62185600, 91165620, 93401290, 93868740] ...
  Inverse  (0): [] ...

📍 Zelle: CRS3035RES100mN3133600E4298400
  Original (8): [5951570, 10673460, 16539510, 78888820, 86441730] ...
  Inverse  (0): [] ...

📍 Zelle: CRS3035RES100mN3133600E4298500
  Original (1): [90738510] ...
  Inverse  (0): [] ...

📍 Zelle: CRS3035RES100mN3133700E4298300
  Original (12): [3384460, 12548580, 38722970, 40922710